<a href="https://colab.research.google.com/github/phuongtvyz/LaDe-Spatiotemporal-Clustering-Thesis/blob/main/MIE4006_Store_Sales_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# 1. KHAI BÁO ĐƯỜNG DẪN DỰ ÁN
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/BusinessForecastingModels"
)

DATA_DIR = PROJECT_DIR / "Data"

OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"


# ============================================================
# 2. KIỂM TRA ĐƯỜNG DẪN
# ============================================================

print("=" * 70)
print("KIỂM TRA CẤU TRÚC DỰ ÁN")
print("=" * 70)

print("PROJECT_DIR:", PROJECT_DIR)
print("PROJECT_DIR tồn tại:", PROJECT_DIR.is_dir())

print("DATA_DIR:", DATA_DIR)
print("DATA_DIR tồn tại:", DATA_DIR.is_dir())


# Nếu không tìm thấy thư mục dự án, dừng chương trình
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục dự án:\n{PROJECT_DIR}\n"
        "Hãy kiểm tra tên thư mục trên Google Drive."
    )

# Nếu không tìm thấy Data, dừng chương trình
if not DATA_DIR.is_dir():
    print("\nCác thư mục trong MyDrive:")

    drive_root = Path("/content/drive/MyDrive")

    for folder in sorted(drive_root.iterdir()):
        if folder.is_dir():
            print("-", repr(folder.name))

    raise FileNotFoundError(
        f"Không tìm thấy thư mục Data tại:\n{DATA_DIR}\n"
        "Hãy kiểm tra tên thư mục và vị trí của Data."
    )


# ============================================================
# 3. TẠO CÁC THƯ MỤC LƯU KẾT QUẢ
# ============================================================

for folder in [
    FIGURES_DIR,
    TABLES_DIR,
    PREDICTIONS_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# 4. LIỆT KÊ CÁC TỆP TRONG DATA
# ============================================================

print("\n" + "=" * 70)
print("DANH SÁCH TỆP TRONG DATA")
print("=" * 70)

data_files = sorted(DATA_DIR.iterdir())

if not data_files:
    print("Thư mục Data chưa có tệp.")
else:
    for file in data_files:
        if file.is_file():
            print(
                f"[TỆP] {file.name} "
                f"({file.stat().st_size / 1024**2:.2f} MB)"
            )
        elif file.is_dir():
            print(f"[THƯ MỤC] {file.name}")

print("\nHoàn tất kiểm tra đường dẫn.")


# ============================================================
# 1. HÀM TÌM TỆP CSV
# ============================================================

def find_csv_file(data_dir, candidate_names):
    for filename in candidate_names:
        file_path = data_dir / filename

        if file_path.is_file():
            return file_path

    raise FileNotFoundError(
        f"Không tìm thấy tệp phù hợp trong {data_dir}\n"
        f"Các tên đã kiểm tra: {candidate_names}\n"
        "Hãy kiểm tra lại tên tệp trên Google Drive."
    )


# ============================================================
# 2. XÁC ĐỊNH ĐƯỜNG DẪN CÁC TỆP
# ============================================================

file_paths = {
    "train": find_csv_file(
        DATA_DIR, ["train.csv"]
    ),

    "test": find_csv_file(
        DATA_DIR,
        ["test.csv", "test(2).csv", "test (2).csv"]
    ),

    "stores": find_csv_file(
        DATA_DIR, ["stores.csv"]
    ),

    "oil": find_csv_file(
        DATA_DIR, ["oil.csv"]
    ),

    "holidays": find_csv_file(
        DATA_DIR, ["holidays_events.csv"]
    ),

    "transactions": find_csv_file(
        DATA_DIR, ["transactions.csv"]
    ),

    "sample_submission": find_csv_file(
        DATA_DIR, ["sample_submission.csv"]
    )
}
# ============================================================
# 3. ĐỌC DỮ LIỆU
# ============================================================

data = {}

for name, path in file_paths.items():
    print(f"Đang đọc: {path.name}")

    if name in [
        "train",
        "test",
        "oil",
        "holidays",
        "transactions"
    ]:
        df = pd.read_csv(
            path,
            parse_dates=["date"]
        )
    else:
        df = pd.read_csv(path)

    data[name] = df

    print(f"  Kích thước: {df.shape}")


# ============================================================
# 4. GÁN BIẾN DỮ LIỆU
# ============================================================

train_df = data["train"]
test_df = data["test"]
stores_df = data["stores"]
oil_df = data["oil"]
holidays_df = data["holidays"]
transactions_df = data["transactions"]
sample_submission_df = data["sample_submission"]

print("\nĐã đọc thành công 7 tệp dữ liệu.")

# ============================================================
# 1. KIỂM TRA KÍCH THƯỚC VÀ CỘT DỮ LIỆU
# ============================================================

print("=" * 70)
print("1. TỔNG QUAN DỮ LIỆU")
print("=" * 70)

for name, df in data.items():
    print(f"\n{name.upper()}")
    print("Số dòng, số cột:", df.shape)
    print("Tên cột:", df.columns.tolist())
    display(df.head(3))


# ============================================================
# 2. KIỂM TRA KHOẢNG THỜI GIAN
# ============================================================

print("\n" + "=" * 70)
print("2. KHOẢNG THỜI GIAN")
print("=" * 70)

for name, df in data.items():
    if "date" in df.columns:
        print(f"\n{name.upper()}")
        print("Ngày đầu tiên:", df["date"].min())
        print("Ngày cuối cùng:", df["date"].max())
        print("Số ngày khác nhau:", df["date"].nunique())
        print("Ngày bị thiếu:", df["date"].isna().sum())


# ============================================================
# 3. KIỂM TRA GIÁ TRỊ THIẾU VÀ DÒNG TRÙNG
# ============================================================

quality_results = []

for name, df in data.items():
    quality_results.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

quality_df = pd.DataFrame(quality_results)

print("\n" + "=" * 70)
print("3. CHẤT LƯỢNG DỮ LIỆU")
print("=" * 70)

display(quality_df)


# ============================================================
# 4. KIỂM TRA KHÓA DỮ LIỆU TRAIN
# ============================================================

key_cols = ["date", "store_nbr", "family"]

print("\n" + "=" * 70)
print("4. KIỂM TRA KHÓA DỮ LIỆU TRAIN")
print("=" * 70)

print("Số cửa hàng:", train_df["store_nbr"].nunique())
print("Số nhóm sản phẩm:", train_df["family"].nunique())
print("Số ngày:", train_df["date"].nunique())

print(
    "Số dòng trùng khóa date-store_nbr-family:",
    train_df.duplicated(subset=key_cols).sum()
)
# ============================================================
# 5. THỐNG KÊ DOANH SỐ
# ============================================================

print("\n" + "=" * 70)
print("5. THỐNG KÊ DOANH SỐ")
print("=" * 70)

display(train_df["sales"].describe().to_frame())

print(
    "Số quan sát sales bằng 0:",
    int((train_df["sales"] == 0).sum())
)

print(
    "Số quan sát sales âm:",
    int((train_df["sales"] < 0).sum())
)
# ============================================================
# 1. LƯU BẢNG CHẤT LƯỢNG DỮ LIỆU
# ============================================================

quality_path = TABLES_DIR / "data_quality_check.csv"

quality_df.to_csv(
    quality_path,
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu:", quality_path)


# ============================================================
# 2. LƯU BẢNG TỔNG QUAN TRAIN VÀ TEST
# ============================================================

summary_rows = []

for name in ["train", "test"]:
    df = data[name]

    summary_rows.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "date_min": df["date"].min(),
        "date_max": df["date"].max(),
        "unique_dates": df["date"].nunique()
    })

summary_df = pd.DataFrame(summary_rows)

summary_path = TABLES_DIR / "dataset_summary.csv"

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

display(summary_df)

print("Đã lưu:", summary_path)

# ============================================================
# CELL 6. PHÂN TÍCH KHÁM PHÁ DỮ LIỆU (EDA)
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. KIỂM TRA PHẠM VI THỜI GIAN VÀ ĐỘ PHỦ DỮ LIỆU
# ------------------------------------------------------------

print("=" * 70)
print("1. KIỂM TRA ĐỘ PHỦ DỮ LIỆU")
print("=" * 70)

n_stores = train_df["store_nbr"].nunique()
n_families = train_df["family"].nunique()
n_dates = train_df["date"].nunique()

expected_rows = n_stores * n_families * n_dates

coverage_summary = pd.DataFrame({
    "metric": [
        "Số cửa hàng",
        "Số nhóm sản phẩm",
        "Số ngày",
        "Số dòng thực tế",
        "Số dòng kỳ vọng nếu lưới đầy đủ",
        "Chênh lệch dòng"
    ],
    "value": [
        n_stores,
        n_families,
        n_dates,
        len(train_df),
        expected_rows,
        expected_rows - len(train_df)
    ]
})

display(coverage_summary)

# Kiểm tra khóa duy nhất
key_cols = ["date", "store_nbr", "family"]

print(
    "\nSố khóa date-store_nbr-family trùng:",
    train_df.duplicated(subset=key_cols).sum()
)

# ------------------------------------------------------------
# 2. PHÂN TÍCH DOANH SỐ BẰNG 0
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. PHÂN TÍCH DOANH SỐ BẰNG 0")
print("=" * 70)

zero_sales_count = int((train_df["sales"] == 0).sum())
total_rows = len(train_df)

zero_sales_summary = pd.DataFrame({
    "metric": [
        "Tổng số quan sát",
        "Số quan sát sales = 0",
        "Số quan sát sales > 0",
        "Tỷ lệ sales = 0 (%)"
    ],
    "value": [
        total_rows,
        zero_sales_count,
        int((train_df["sales"] > 0).sum()),
        round(zero_sales_count / total_rows * 100, 2)
    ]
})

display(zero_sales_summary)

# ------------------------------------------------------------
# 3. TỶ LỆ DOANH SỐ BẰNG 0 THEO NHÓM SẢN PHẨM
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. DOANH SỐ BẰNG 0 THEO NHÓM SẢN PHẨM")
print("=" * 70)

family_summary = (
    train_df
    .groupby("family", observed=True)
    .agg(
        observations=("sales", "size"),
        total_sales=("sales", "sum"),
        mean_sales=("sales", "mean"),
        median_sales=("sales", "median"),
        zero_sales_count=("sales", lambda x: (x == 0).sum())
    )
    .reset_index()
)

family_summary["zero_sales_pct"] = (
    family_summary["zero_sales_count"]
    / family_summary["observations"]
    * 100
)

family_summary = family_summary.sort_values(
    "zero_sales_pct",
    ascending=False
)

display(family_summary)

# ------------------------------------------------------------
# 4. DOANH SỐ THEO THỜI GIAN
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. DOANH SỐ THEO THỜI GIAN")
print("=" * 70)

daily_sales = (
    train_df
    .groupby("date", as_index=False)
    .agg(
        total_sales=("sales", "sum"),
        mean_sales=("sales", "mean"),
        zero_sales_count=("sales", lambda x: (x == 0).sum())
    )
)

daily_sales["zero_sales_pct"] = (
    daily_sales["zero_sales_count"]
    / (n_stores * n_families)
    * 100
)

display(daily_sales.head())
display(daily_sales.tail())

# ------------------------------------------------------------
# 5. KIỂM TRA DỮ LIỆU GIÁ DẦU
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("5. KIỂM TRA GIÁ DẦU")
print("=" * 70)

oil_missing = int(oil_df["dcoilwtico"].isna().sum())
oil_total = len(oil_df)

print("Tổng số dòng:", oil_total)
print("Số giá trị giá dầu thiếu:", oil_missing)
print(
    "Tỷ lệ thiếu (%):",
    round(oil_missing / oil_total * 100, 2)
)

print("\nCác dòng giá dầu bị thiếu:")
display(
    oil_df.loc[
        oil_df["dcoilwtico"].isna()
    ].head(20)
)

# ------------------------------------------------------------
# 6. KIỂM TRA NGÀY CÓ GIAO DỊCH
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("6. KIỂM TRA PHẠM VI DỮ LIỆU GIAO DỊCH")
print("=" * 70)

print("Ngày đầu tiên:", transactions_df["date"].min())
print("Ngày cuối cùng:", transactions_df["date"].max())
print("Số ngày có dữ liệu:", transactions_df["date"].nunique())

print(
    "Số ngày trong train không xuất hiện trong transactions:",
    len(
        set(train_df["date"].unique())
        - set(transactions_df["date"].unique())
    )
)

print(
    "Ngày cuối train:",
    train_df["date"].max()
)

print(
    "Ngày đầu test:",
    test_df["date"].min()
)

print(
    "Transactions có dữ liệu sau ngày cuối train:",
    (
        transactions_df["date"].max()
        > train_df["date"].max()
    )
)

# ------------------------------------------------------------
# 7. LƯU KẾT QUẢ
# ------------------------------------------------------------

coverage_summary.to_csv(
    TABLES_DIR / "eda_data_coverage.csv",
    index=False,
    encoding="utf-8-sig"
)

zero_sales_summary.to_csv(
    TABLES_DIR / "eda_zero_sales_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

family_summary.to_csv(
    TABLES_DIR / "eda_sales_by_family.csv",
    index=False,
    encoding="utf-8-sig"
)

daily_sales.to_csv(
    TABLES_DIR / "eda_daily_sales.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã lưu 4 bảng EDA vào:")
print(TABLES_DIR)
# CELL 7. THIẾT KẾ TẬP VALIDATION THEO THỜI GIAN

import pandas as pd

VALIDATION_START = pd.Timestamp("2017-07-16")
VALIDATION_END = pd.Timestamp("2017-07-31")

TEST_START = pd.Timestamp("2017-08-16")
TEST_END = pd.Timestamp("2017-08-31")

# Chuẩn hóa kiểu dữ liệu ngày
train_df["date"] = pd.to_datetime(train_df["date"])
test_df["date"] = pd.to_datetime(test_df["date"])

# Chia dữ liệu theo thời gian
train_model_df = train_df[
    train_df["date"] < VALIDATION_START
].copy()

validation_df = train_df[
    (train_df["date"] >= VALIDATION_START)
    & (train_df["date"] <= VALIDATION_END)
].copy()

# Tập test được giữ riêng, không dùng để chọn mô hình
test_model_df = test_df[
    (test_df["date"] >= TEST_START)
    & (test_df["date"] <= TEST_END)
].copy()

# Tóm tắt các tập dữ liệu
split_summary = pd.DataFrame([
    {
        "dataset": "Train for model selection",
        "rows": len(train_model_df),
        "start_date": train_model_df["date"].min(),
        "end_date": train_model_df["date"].max(),
        "unique_dates": train_model_df["date"].nunique()
    },
    {
        "dataset": "Validation",
        "rows": len(validation_df),
        "start_date": validation_df["date"].min(),
        "end_date": validation_df["date"].max(),
        "unique_dates": validation_df["date"].nunique()
    },
    {
        "dataset": "Final test",
        "rows": len(test_model_df),
        "start_date": test_model_df["date"].min(),
        "end_date": test_model_df["date"].max(),
        "unique_dates": test_model_df["date"].nunique()
    }
])

display(split_summary)

# Kiểm tra điều kiện chia dữ liệu
assert train_model_df["date"].max() < validation_df["date"].min()
assert validation_df["date"].max() < test_model_df["date"].min()
assert validation_df["date"].nunique() == 16
assert test_model_df["date"].nunique() == 16

print("Kiểm tra chia dữ liệu: PASS")

# Lưu bảng kiểm tra
split_summary.to_csv(
    TABLES_DIR / "time_split_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu:", TABLES_DIR / "time_split_summary.csv")

# CELL 8. KIỂM TRA CÁC NGÀY LỊCH KHÔNG XUẤT HIỆN

import pandas as pd

# Tạo toàn bộ lịch ngày từ đầu đến cuối tập train gốc
calendar_dates = pd.date_range(
    start=train_df["date"].min(),
    end=train_df["date"].max(),
    freq="D"
)

# Những ngày có trong lịch nhưng không có trong train_df
observed_dates = pd.DatetimeIndex(
    train_df["date"].drop_duplicates()
)

missing_dates = calendar_dates.difference(observed_dates)

print("Tổng số ngày theo lịch:", len(calendar_dates))
print("Số ngày có trong train_df:", len(observed_dates))
print("Số ngày không xuất hiện:", len(missing_dates))
print("\nDanh sách ngày không xuất hiện:")
display(pd.DataFrame({"missing_date": missing_dates}))

# Kiểm tra số dòng theo từng ngày trong train gốc
daily_row_counts = (
    train_df.groupby("date")
    .size()
    .reindex(calendar_dates, fill_value=0)
)

print("\nPhân bố số dòng theo ngày:")
display(daily_row_counts.value_counts().rename_axis(
    "rows_per_date"
).reset_index(name="number_of_dates").sort_values("rows_per_date"))

# Kiểm tra riêng tập train dùng để huấn luyện
expected_rows_per_date = (
    train_df["store_nbr"].nunique()
    * train_df["family"].nunique()
)

train_dates_before_validation = pd.date_range(
    start=train_model_df["date"].min(),
    end=train_model_df["date"].max(),
    freq="D"
)

missing_dates_before_validation = train_dates_before_validation.difference(
    pd.DatetimeIndex(train_model_df["date"].unique())
)

print("\nSố ngày theo lịch trước validation:",
      len(train_dates_before_validation))
print("Số ngày có dữ liệu trước validation:",
      train_model_df["date"].nunique())
print("Số ngày thiếu trước validation:",
      len(missing_dates_before_validation))
print("Số dòng kỳ vọng mỗi ngày:",
      expected_rows_per_date)

display(pd.DataFrame({
    "missing_date_before_validation": missing_dates_before_validation
}))
# CELL 9. BASELINE SEASONAL NAIVE

import pandas as pd
import numpy as np

KEYS = ["store_nbr", "family"]

# Tạo bảng tra cứu doanh số thực tế theo cửa hàng, nhóm sản phẩm và ngày
sales_lookup = train_df[
    ["date", "store_nbr", "family", "sales"]
].copy()

sales_lookup["date"] = pd.to_datetime(sales_lookup["date"])

# Ngày dùng làm căn cứ dự báo: lùi đúng 7 ngày lịch
validation_pred = validation_df[
    ["id", "date", "store_nbr", "family", "sales"]
].copy()

validation_pred["date"] = pd.to_datetime(validation_pred["date"])
validation_pred["lag_date"] = (
    validation_pred["date"] - pd.Timedelta(days=7)
)

# Tra cứu doanh số của đúng cửa hàng, nhóm sản phẩm và ngày lag
validation_pred = validation_pred.merge(
    sales_lookup.rename(
        columns={
            "date": "lag_date",
            "sales": "seasonal_naive_prediction"
        }
    ),
    on=["lag_date", "store_nbr", "family"],
    how="left",
    validate="many_to_one"
)

# Kiểm tra số dự báo không tìm được dữ liệu lịch sử
missing_baseline = validation_pred[
    "seasonal_naive_prediction"
].isna().sum()

print("Số quan sát validation:", len(validation_pred))
print("Số dự báo Seasonal Naive bị thiếu:", missing_baseline)

display(
    validation_pred[
        [
            "date",
            "store_nbr",
            "family",
            "sales",
            "lag_date",
            "seasonal_naive_prediction"
        ]
    ].head(10)
)

# Lưu dự báo baseline
validation_pred.to_csv(
    PREDICTIONS_DIR / "validation_seasonal_naive.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Đã lưu:",
    PREDICTIONS_DIR / "validation_seasonal_naive.csv"
)
# CELL 10. ĐÁNH GIÁ BASELINE SEASONAL NAIVE

import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error
)

# Kiểm tra dữ liệu đầu vào
actual = validation_pred["sales"].to_numpy(dtype=float)

predicted = validation_pred[
    "seasonal_naive_prediction"
].to_numpy(dtype=float)

assert len(actual) == len(predicted), (
    "Số lượng giá trị thực tế và dự báo không khớp."
)

assert np.isfinite(actual).all(), (
    "Dữ liệu sales có giá trị không hữu hạn."
)

assert np.isfinite(predicted).all(), (
    "Dự báo có giá trị không hữu hạn."
)

assert (actual >= 0).all(), (
    "Có doanh số thực tế âm; cần kiểm tra trước khi tính RMSLE."
)

assert (predicted >= 0).all(), (
    "Có dự báo âm; cần kiểm tra trước khi tính RMSLE."
)

# Tính các chỉ số đánh giá
mae = mean_absolute_error(actual, predicted)

rmse = np.sqrt(
    mean_squared_error(actual, predicted)
)

rmsle = np.sqrt(
    mean_squared_log_error(actual, predicted)
)

baseline_metrics = pd.DataFrame({
    "model": ["Seasonal Naive"],
    "MAE": [mae],
    "RMSE": [rmse],
    "RMSLE": [rmsle],
    "validation_rows": [len(actual)]
})

display(baseline_metrics.round(4))

# Lưu kết quả
baseline_metrics.to_csv(
    TABLES_DIR / "baseline_validation_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Đã lưu:",
    TABLES_DIR / "baseline_validation_metrics.csv"
)
# CELL 11. ĐÁNH GIÁ BASELINE THEO NHÓM SẢN PHẨM

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error
)

def calculate_metrics(group):
    y_true = group["sales"].to_numpy(dtype=float)
    y_pred = group[
        "seasonal_naive_prediction"
    ].to_numpy(dtype=float)

    return pd.Series({
        "observations": len(group),
        "actual_mean_sales": y_true.mean(),
        "zero_sales_pct": (y_true == 0).mean() * 100,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "RMSLE": np.sqrt(
            mean_squared_log_error(y_true, y_pred)
        )
    })

family_metrics = (
    validation_pred
    .groupby("family")
    .apply(calculate_metrics, include_groups=False)
    .reset_index()
    .sort_values("RMSLE", ascending=False)
)

display(family_metrics.round(4))

family_metrics.to_csv(
    TABLES_DIR / "baseline_metrics_by_family.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Đã lưu:",
    TABLES_DIR / "baseline_metrics_by_family.csv"
)
# CELL 12. KIỂM TRA THƯ VIỆN LIGHTGBM

import importlib.util
import sklearn
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

if importlib.util.find_spec("lightgbm") is None:
    %pip install -q lightgbm

import lightgbm as lgb

print("LightGBM:", lgb.__version__)
print("Kiểm tra thư viện: PASS")
# CELL 13. KIỂM TRA ĐẦU VÀO CHO GLOBAL LIGHTGBM

required_objects = [
    "train_df",
    "test_df",
    "stores_df",
    "train_model_df",
    "validation_df",
    "test_model_df",
    "validation_pred",
    "PROJECT_DIR",
    "TABLES_DIR",
    "PREDICTIONS_DIR"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Thiếu các biến cần thiết: {missing_objects}. "
        "Hãy kiểm tra và chạy lại những cell tạo các biến này."
    )

print("Số dòng train_model_df:", len(train_model_df))
print("Số dòng validation_df:", len(validation_df))
print("Số dòng test_model_df:", len(test_model_df))

print("\nKhoảng thời gian:")
print(
    "Train:",
    train_model_df["date"].min(),
    "→",
    train_model_df["date"].max()
)
print(
    "Validation:",
    validation_df["date"].min(),
    "→",
    validation_df["date"].max()
)
print(
    "Test:",
    test_model_df["date"].min(),
    "→",
    test_model_df["date"].max()
)

print("\nSố cửa hàng trong train:", train_model_df["store_nbr"].nunique())
print("Số nhóm sản phẩm trong train:", train_model_df["family"].nunique())

print("\nSố cửa hàng trong validation:", validation_df["store_nbr"].nunique())
print("Số nhóm sản phẩm trong validation:", validation_df["family"].nunique())

print("\nSố dòng validation có dự báo Seasonal Naive:",
      len(validation_pred))

assert len(validation_df) == 28_512
assert len(test_model_df) == 28_512
assert len(validation_pred) == len(validation_df)

assert train_model_df["date"].max() < validation_df["date"].min()
assert validation_df["date"].max() < test_model_df["date"].min()

assert train_model_df["date"].max() == pd.Timestamp("2017-07-15")
assert validation_df["date"].min() == pd.Timestamp("2017-07-16")
assert validation_df["date"].max() == pd.Timestamp("2017-07-31")
assert test_model_df["date"].min() == pd.Timestamp("2017-08-16")
assert test_model_df["date"].max() == pd.Timestamp("2017-08-31")

print("\nKiểm tra đầu vào Global LightGBM: PASS")

# CELL 14. CHUẨN BỊ ĐẶC TRƯNG CỬA HÀNG, SẢN PHẨM VÀ LỊCH

import pandas as pd
import numpy as np

# Sao chép dữ liệu, tránh sửa trực tiếp các DataFrame gốc
model_train_base = train_model_df.copy()
model_validation_base = validation_df.copy()
model_test_base = test_model_df.copy()

# Chuẩn hóa kiểu ngày
for df in [
    model_train_base,
    model_validation_base,
    model_test_base
]:
    df["date"] = pd.to_datetime(df["date"])

# Kiểm tra khóa cửa hàng trong bảng stores
assert stores_df["store_nbr"].is_unique, (
    "stores_df có store_nbr trùng."
)

store_columns = [
    "store_nbr",
    "city",
    "state",
    "type",
    "cluster"
]

# Ghép thông tin cửa hàng
model_train_base = model_train_base.merge(
    stores_df[store_columns],
    on="store_nbr",
    how="left",
    validate="many_to_one"
)

model_validation_base = model_validation_base.merge(
    stores_df[store_columns],
    on="store_nbr",
    how="left",
    validate="many_to_one"
)

model_test_base = model_test_base.merge(
    stores_df[store_columns],
    on="store_nbr",
    how="left",
    validate="many_to_one"
)

# Tạo đặc trưng lịch
def add_calendar_features(df):
    df = df.copy()

    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_month"] = df["date"].dt.day
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    df["is_weekend"] = (
        df["day_of_week"] >= 5
    ).astype("int8")

    return df

model_train_base = add_calendar_features(model_train_base)
model_validation_base = add_calendar_features(model_validation_base)
model_test_base = add_calendar_features(model_test_base)

# Kiểm tra
print("Train:", model_train_base.shape)
print("Validation:", model_validation_base.shape)
print("Test:", model_test_base.shape)

print("\nSố giá trị thiếu sau khi ghép thông tin cửa hàng:")
display(
    pd.DataFrame({
        "dataset": ["train", "validation", "test"],
        "missing_store_info": [
            model_train_base[["city", "state", "type", "cluster"]]
                .isna().sum().sum(),
            model_validation_base[["city", "state", "type", "cluster"]]
                .isna().sum().sum(),
            model_test_base[["city", "state", "type", "cluster"]]
                .isna().sum().sum()
        ]
    })
)

print("\nCác đặc trưng lịch:")
display(
    model_train_base[
        [
            "date",
            "store_nbr",
            "family",
            "day_of_week",
            "day_of_month",
            "month",
            "year",
            "is_weekend"
        ]
    ].head()
)

assert len(model_train_base) == len(train_model_df)
assert len(model_validation_base) == len(validation_df)
assert len(model_test_base) == len(test_model_df)

assert model_train_base[
    ["city", "state", "type", "cluster"]
].isna().sum().sum() == 0

assert model_validation_base[
    ["city", "state", "type", "cluster"]
].isna().sum().sum() == 0

assert model_test_base[
    ["city", "state", "type", "cluster"]
].isna().sum().sum() == 0

print("\nKiểm tra đặc trưng cơ bản: PASS")
# CELL 15. TẠO LAG VÀ ROLLING FEATURES THEO NGÀY LỊCH

import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Chuẩn bị lịch sử doanh số
# --------------------------------------------------

history_df = train_df[
    ["date", "store_nbr", "family", "sales"]
].copy()

history_df["date"] = pd.to_datetime(history_df["date"])

# Mỗi tổ hợp ngày - cửa hàng - nhóm sản phẩm phải duy nhất
assert not history_df.duplicated(
    ["date", "store_nbr", "family"]
).any(), "Phát hiện khóa ngày-cửa hàng-nhóm sản phẩm trùng."

# --------------------------------------------------
# 2. Tạo lịch sử theo từng cửa hàng và nhóm sản phẩm
#    Bổ sung ngày lịch còn thiếu dưới dạng NaN,
#    KHÔNG gán doanh số bằng 0.
# --------------------------------------------------

date_min = history_df["date"].min()
date_max = history_df["date"].max()

all_dates = pd.date_range(
    start=date_min,
    end=date_max,
    freq="D"
)

feature_frames = []

for (store_nbr, family), group in history_df.groupby(
    ["store_nbr", "family"],
    sort=False
):
    series = (
        group.set_index("date")["sales"]
        .sort_index()
        .reindex(all_dates)
    )

    # Chỉ dùng giá trị quá khứ.
    # Các ngày thiếu vẫn là NaN.
    features = pd.DataFrame(index=all_dates)

    features["store_nbr"] = store_nbr
    features["family"] = family

    features["lag_7"] = series.shift(7)
    features["lag_14"] = series.shift(14)
    features["lag_28"] = series.shift(28)

    features["rolling_mean_7"] = (
        series.shift(1)
        .rolling(window=7, min_periods=1)
        .mean()
    )

    features["rolling_mean_14"] = (
        series.shift(1)
        .rolling(window=14, min_periods=1)
        .mean()
    )

    features["rolling_mean_28"] = (
        series.shift(1)
        .rolling(window=28, min_periods=1)
        .mean()
    )

    features.index.name = "date"
    feature_frames.append(features.reset_index())

# Ghép đặc trưng của các tổ hợp
history_features = pd.concat(
    feature_frames,
    ignore_index=True
)

# --------------------------------------------------
# 3. Ghép đặc trưng vào tập huấn luyện
# --------------------------------------------------

model_train_features = model_train_base.merge(
    history_features,
    on=["date", "store_nbr", "family"],
    how="left",
    validate="one_to_one"
)

# --------------------------------------------------
# 4. Kiểm tra kết quả
# --------------------------------------------------

lag_cols = [
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28"
]

print("Kích thước model_train_features:",
      model_train_features.shape)

print("\nSố dòng ban đầu:", len(model_train_base))
print("Số dòng sau khi ghép:", len(model_train_features))

print("\nTỷ lệ thiếu của các đặc trưng:")
display(
    model_train_features[lag_cols]
    .isna()
    .mean()
    .mul(100)
    .round(4)
    .rename("missing_pct")
    .to_frame()
)

print("\nVí dụ đặc trưng:")
display(
    model_train_features[
        [
            "date",
            "store_nbr",
            "family",
            "sales",
            "lag_7",
            "lag_14",
            "lag_28",
            "rolling_mean_7",
            "rolling_mean_14",
            "rolling_mean_28"
        ]
    ].tail(10)
)

# Kiểm tra tính toàn vẹn sau khi ghép
assert len(model_train_features) == len(model_train_base), (
    "Số dòng thay đổi sau khi ghép đặc trưng."
)

assert not model_train_features.duplicated(
    ["date", "store_nbr", "family"]
).any(), "Xuất hiện khóa trùng sau khi ghép đặc trưng."

assert model_train_features["sales"].notna().all(), (
    "Biến mục tiêu sales bị thiếu."
)

print("\nKiểm tra lag/rolling features: PASS")
# CELL 16. CHUẨN BỊ MA TRẬN ĐẶC TRƯNG HUẤN LUYỆN

import pandas as pd
import numpy as np

# Biến mục tiêu
TARGET = "sales"

# Biến phân loại
CAT_FEATURES = [
    "family",
    "city",
    "state",
    "type"
]

# Biến số và biến lịch
NUM_FEATURES = [
    "store_nbr",
    "cluster",
    "day_of_week",
    "day_of_month",
    "month",
    "year",
    "is_weekend",
    "onpromotion",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28"
]

FEATURES = CAT_FEATURES + NUM_FEATURES

# Kiểm tra sự tồn tại của các cột
missing_features = [
    col for col in FEATURES + [TARGET]
    if col not in model_train_features.columns
]

if missing_features:
    raise ValueError(
        f"Thiếu cột cần thiết: {missing_features}"
    )

# Tạo bản sao cho huấn luyện
train_ml = model_train_features[
    FEATURES + [TARGET]
].copy()

# Chuyển biến phân loại về category
for col in CAT_FEATURES:
    train_ml[col] = train_ml[col].astype("category")

# Kiểm tra kiểu dữ liệu
print("Kích thước train_ml:", train_ml.shape)

print("\nKiểu dữ liệu:")
display(train_ml.dtypes.rename("dtype").to_frame())

print("\nSố giá trị thiếu theo đặc trưng:")
display(
    train_ml[FEATURES]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

print("\nThống kê biến mục tiêu:")
display(train_ml[TARGET].describe().to_frame())

# Kiểm tra dữ liệu mục tiêu
assert train_ml[TARGET].notna().all(), (
    "Biến mục tiêu sales có giá trị thiếu."
)

assert np.isfinite(
    train_ml[TARGET].to_numpy(dtype=float)
).all(), "Biến mục tiêu chứa giá trị không hữu hạn."

assert (train_ml[TARGET] >= 0).all(), (
    "Phát hiện doanh số âm."
)

print("\nKiểm tra ma trận đặc trưng: PASS")

# CELL 16 — Chuẩn bị ma trận đặc trưng cho Global LightGBM

TARGET = "sales"

# Các biến mang tính phân loại hoặc định danh
CAT_FEATURES = [
    "store_nbr",
    "family",
    "city",
    "state",
    "type",
    "cluster"
]

# Các biến số có ý nghĩa về thời gian, khuyến mãi và lịch sử doanh số
NUM_FEATURES = [
    "day_of_week",
    "day_of_month",
    "month",
    "year",
    "is_weekend",
    "onpromotion",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28"
]

FEATURES = CAT_FEATURES + NUM_FEATURES

# Kiểm tra các cột cần thiết
required_columns = FEATURES + [TARGET]
missing_columns = [
    col for col in required_columns
    if col not in model_train_features.columns
]

assert not missing_columns, (
    f"Thiếu các cột: {missing_columns}"
)

# Tạo ma trận huấn luyện
train_ml = model_train_features[required_columns].copy()

# Chuẩn hóa kiểu dữ liệu của các biến phân loại
for col in CAT_FEATURES:
    train_ml[col] = train_ml[col].astype("category")

# Kiểm tra dữ liệu
print("Kích thước train_ml:", train_ml.shape)
print("\nSố lượng đặc trưng:", len(FEATURES))
print("Số biến phân loại:", len(CAT_FEATURES))
print("Số biến số:", len(NUM_FEATURES))

print("\nKiểu dữ liệu:")
print(train_ml.dtypes)

print("\nSố giá trị thiếu theo cột:")
print(train_ml.isna().sum()[train_ml.isna().sum() > 0])

print("\nThống kê biến mục tiêu:")
print(train_ml[TARGET].describe())

# Kiểm tra biến mục tiêu
assert train_ml[TARGET].notna().all(), (
    "Biến mục tiêu có giá trị thiếu."
)

assert np.isfinite(train_ml[TARGET]).all(), (
    "Biến mục tiêu có giá trị vô hạn hoặc không hữu hạn."
)

assert (train_ml[TARGET] >= 0).all(), (
    "Phát hiện doanh số âm."
)

# Cho phép NaN để LightGBM xử lý, nhưng không cho phép +inf hoặc -inf
numeric_values = train_ml[NUM_FEATURES].to_numpy(dtype=float)

assert not np.isinf(numeric_values).any(), (
    "Có giá trị +inf hoặc -inf trong các đặc trưng số."
)

# CELL 17 — Huấn luyện Global LightGBM

from pathlib import Path
import lightgbm as lgb
import pandas as pd
import numpy as np
import json
import time

# 1. Xác định thư mục lưu kết quả
MODEL_DIR = OUTPUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 2. Chuẩn bị dữ liệu
X_train = train_ml[FEATURES].copy()
y_train = train_ml[TARGET].copy()

# 3. Khai báo mô hình
lgb_model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# 4. Huấn luyện
start_time = time.time()

lgb_model.fit(
    X_train,
    y_train,
    categorical_feature=CAT_FEATURES
)

training_seconds = time.time() - start_time

# 5. Lưu mô hình
model_path = MODEL_DIR / "global_lightgbm.txt"
lgb_model.booster_.save_model(str(model_path))

# 6. Lưu thông tin cấu hình
model_config = {
    "model": "Global LightGBM",
    "objective": "regression",
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 100,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "categorical_features": CAT_FEATURES,
    "numeric_features": NUM_FEATURES,
    "target": TARGET,
    "training_rows": int(len(X_train)),
    "training_seconds": round(training_seconds, 2)
}

with open(MODEL_DIR / "global_lightgbm_config.json", "w") as f:
    json.dump(model_config, f, indent=2)

# 7. Kiểm tra kết quả
print("Huấn luyện hoàn tất.")
print("Số dòng huấn luyện:", len(X_train))
print("Số đặc trưng:", len(FEATURES))
print("Thời gian huấn luyện (giây):", round(training_seconds, 2))
print("Model path:", model_path)
print("Số cây đã huấn luyện:", lgb_model.booster_.num_trees())

print("\nPASS: Global LightGBM đã được huấn luyện và lưu.")
# CELL 18 — Recursive validation forecasting (16 days)

import pandas as pd
import numpy as np
import time

VALIDATION_START = pd.Timestamp("2017-07-16")
VALIDATION_END = pd.Timestamp("2017-07-31")
HISTORY_CUTOFF = VALIDATION_START - pd.Timedelta(days=1)

# Chỉ lấy doanh số đã quan sát trước validation
observed_history = train_df.loc[
    pd.to_datetime(train_df["date"]) <= HISTORY_CUTOFF,
    ["date", "store_nbr", "family", "sales"]
].copy()

observed_history["date"] = pd.to_datetime(
    observed_history["date"]
)

assert observed_history["date"].max() == HISTORY_CUTOFF

# Tạo bảng lịch sử dạng wide: mỗi cột là một chuỗi
sales_history = observed_history.pivot(
    index="date",
    columns=["store_nbr", "family"],
    values="sales"
)

# Giữ lịch liên tục, ngày không có dữ liệu là NaN
sales_history = sales_history.asfreq("D")

# Dữ liệu validation đã có calendar/store features
validation_recursive = model_validation_base.copy()
validation_recursive["date"] = pd.to_datetime(
    validation_recursive["date"]
)

assert validation_recursive["date"].min() == VALIDATION_START
assert validation_recursive["date"].max() == VALIDATION_END

prediction_batches = []

start_time = time.time()

for current_date in pd.date_range(
    VALIDATION_START, VALIDATION_END, freq="D"
):
    day_df = validation_recursive.loc[
        validation_recursive["date"] == current_date
    ].copy()

    # Lấy dữ liệu lịch sử đúng đến ngày trước ngày dự báo
    past = sales_history.loc[
        :current_date - pd.Timedelta(days=1)
    ]

    # Tạo lag theo ngày lịch
    for lag in [7, 14, 28]:
        lag_date = current_date - pd.Timedelta(days=lag)

        lag_values = past.loc[lag_date]

        lag_lookup = lag_values.rename(
            f"lag_{lag}"
        ).reset_index()

        day_df = day_df.merge(
            lag_lookup,
            on=["store_nbr", "family"],
            how="left",
            validate="one_to_one"
        )

    # Tạo rolling mean từ lịch sử thực tế + dự báo trước đó
    for window in [7, 14, 28]:
        rolling_values = past.tail(window).mean(
            axis=0,
            skipna=True
        )

        rolling_lookup = rolling_values.rename(
            f"rolling_mean_{window}"
        ).reset_index()

        day_df = day_df.merge(
            rolling_lookup,
            on=["store_nbr", "family"],
            how="left",
            validate="one_to_one"
        )

    # Chuẩn hóa category giống tập huấn luyện
    for col in CAT_FEATURES:
        day_df[col] = pd.Categorical(
            day_df[col],
            categories=X_train[col].cat.categories
        )

    # Dự báo
    day_predictions = lgb_model.predict(
        day_df[FEATURES]
    )

    # Doanh số không được âm
    day_predictions = np.maximum(day_predictions, 0)

    day_df["lightgbm_prediction"] = day_predictions

    prediction_batches.append(
        day_df[
            [
                "id",
                "date",
                "store_nbr",
                "family",
                "sales",
                "lightgbm_prediction"
            ]
        ].copy()
    )

    # Cập nhật lịch sử bằng dự báo, KHÔNG bằng sales thực tế
    predicted_series = day_df.set_index(
        ["store_nbr", "family"]
    )["lightgbm_prediction"]

    sales_history.loc[current_date] = (
        predicted_series.reindex(sales_history.columns)
    )

    print(
        f"Đã dự báo: {current_date.date()} "
        f"| {len(day_df)} chuỗi"
    )

# Tổng hợp
validation_lgb_predictions = pd.concat(
    prediction_batches,
    ignore_index=True
)

assert len(validation_lgb_predictions) == len(validation_df)
assert validation_lgb_predictions["lightgbm_prediction"].notna().all()
assert (validation_lgb_predictions["lightgbm_prediction"] >= 0).all()

prediction_path = (
    PREDICTIONS_DIR / "validation_global_lightgbm.csv"
)

validation_lgb_predictions.to_csv(
    prediction_path,
    index=False
)

print("\nPASS: Recursive validation forecast hoàn tất.")
print("Số dòng dự báo:", len(validation_lgb_predictions))
print(
    "Thời gian dự báo (giây):",
    round(time.time() - start_time, 2)
)
print("Đường dẫn lưu:", prediction_path)

display(validation_lgb_predictions.head())

# CELL 19 — Đánh giá Global LightGBM và Seasonal Naive

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import mean_squared_log_error

# 1. Đọc kết quả của hai mô hình
baseline_path = (
    PREDICTIONS_DIR / "validation_seasonal_naive.csv"
)

lightgbm_path = (
    PREDICTIONS_DIR / "validation_global_lightgbm.csv"
)

baseline_pred = pd.read_csv(baseline_path)
lightgbm_pred = pd.read_csv(lightgbm_path)

# 2. Kiểm tra cột bắt buộc
assert {
    "id", "sales", "seasonal_naive_prediction"
}.issubset(baseline_pred.columns)

assert {
    "id", "sales", "lightgbm_prediction"
}.issubset(lightgbm_pred.columns)

assert baseline_pred["id"].is_unique
assert lightgbm_pred["id"].is_unique

# 3. Ghép dự báo
comparison_df = baseline_pred[
    ["id", "sales", "seasonal_naive_prediction"]
].merge(
    lightgbm_pred[
        ["id", "sales", "lightgbm_prediction"]
    ],
    on="id",
    how="inner",
    suffixes=("_baseline", "_lightgbm"),
    validate="one_to_one"
)

assert len(comparison_df) == 28512

assert np.allclose(
    comparison_df["sales_baseline"],
    comparison_df["sales_lightgbm"]
)

comparison_df = comparison_df.rename(
    columns={"sales_baseline": "sales"}
).drop(columns=["sales_lightgbm"])

# 4. Hàm tính chỉ số đánh giá
def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    assert np.isfinite(y_true).all()
    assert np.isfinite(y_pred).all()
    assert (y_true >= 0).all()
    assert (y_pred >= 0).all()

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "RMSLE": np.sqrt(mean_squared_log_error(y_true, y_pred))
    }

# 5. Tính kết quả hai mô hình
baseline_metrics = calculate_metrics(
    comparison_df["sales"],
    comparison_df["seasonal_naive_prediction"]
)

lightgbm_metrics = calculate_metrics(
    comparison_df["sales"],
    comparison_df["lightgbm_prediction"]
)

# 6. Bảng so sánh
comparison_metrics = pd.DataFrame([
    {"model": "Seasonal Naive", **baseline_metrics},
    {"model": "Global LightGBM", **lightgbm_metrics}
])

# 7. Tính tỷ lệ cải thiện của LightGBM so với baseline
improvement = {}

for metric in ["MAE", "RMSE", "RMSLE"]:
    improvement[metric] = (
        (baseline_metrics[metric] - lightgbm_metrics[metric])
        / baseline_metrics[metric]
        * 100
    )

improvement_df = pd.DataFrame([
    {
        "metric": metric,
        "baseline": baseline_metrics[metric],
        "lightgbm": lightgbm_metrics[metric],
        "improvement_pct": improvement[metric]
    }
    for metric in ["MAE", "RMSE", "RMSLE"]
])

# 8. Lưu kết quả
comparison_metrics.to_csv(
    TABLES_DIR / "model_comparison_metrics.csv",
    index=False
)

improvement_df.to_csv(
    TABLES_DIR / "model_improvement_summary.csv",
    index=False
)

comparison_df.to_csv(
    PREDICTIONS_DIR / "validation_model_comparison.csv",
    index=False
)

print("BẢNG 1 — KẾT QUẢ ĐÁNH GIÁ")
display(comparison_metrics.round(4))

print("\nBẢNG 2 — TỶ LỆ CẢI THIỆN CỦA LIGHTGBM")
display(improvement_df.round(4))

print("\nPASS: Đã đánh giá và lưu kết quả so sánh.")
# CELL 20 — Error analysis by family and zero-sales segments

import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error
)

# 1. Đọc kết quả dự báo đã lưu
baseline = pd.read_csv(
    PREDICTIONS_DIR / "validation_seasonal_naive.csv"
)

lightgbm = pd.read_csv(
    PREDICTIONS_DIR / "validation_global_lightgbm.csv"
)

# 2. Ghép dữ liệu theo id
analysis_df = lightgbm[
    ["id", "date", "store_nbr", "family",
     "sales", "lightgbm_prediction"]
].merge(
    baseline[["id", "seasonal_naive_prediction"]],
    on="id",
    how="inner",
    validate="one_to_one"
)

assert len(analysis_df) == 28512
assert analysis_df["id"].is_unique

# 3. Hàm tính sai số
def error_metrics(df, prediction_col):
    y = df["sales"].to_numpy()
    p = df[prediction_col].to_numpy()

    return {
        "MAE": mean_absolute_error(y, p),
        "RMSE": np.sqrt(mean_squared_error(y, p)),
        "RMSLE": np.sqrt(mean_squared_log_error(y, p))
    }

# 4. Phân tích theo nhóm sản phẩm
family_results = []

for family, group in analysis_df.groupby("family"):
    baseline_m = error_metrics(
        group, "seasonal_naive_prediction"
    )
    lightgbm_m = error_metrics(
        group, "lightgbm_prediction"
    )

    family_results.append({
        "family": family,
        "n": len(group),
        "zero_sales_pct": (group["sales"] == 0).mean() * 100,
        "baseline_MAE": baseline_m["MAE"],
        "lightgbm_MAE": lightgbm_m["MAE"],
        "baseline_RMSLE": baseline_m["RMSLE"],
        "lightgbm_RMSLE": lightgbm_m["RMSLE"],
        "MAE_improvement_pct": (
            (baseline_m["MAE"] - lightgbm_m["MAE"])
            / baseline_m["MAE"] * 100
            if baseline_m["MAE"] > 0 else np.nan
        ),
        "RMSLE_improvement_pct": (
            (baseline_m["RMSLE"] - lightgbm_m["RMSLE"])
            / baseline_m["RMSLE"] * 100
            if baseline_m["RMSLE"] > 0 else np.nan
        )
    })

family_analysis = pd.DataFrame(family_results)

# 5. Phân tích theo doanh số thực tế = 0 và > 0
segment_results = []

for segment, group in [
    ("Zero sales", analysis_df[analysis_df["sales"] == 0]),
    ("Positive sales", analysis_df[analysis_df["sales"] > 0])
]:
    for model_name, pred_col in [
        ("Seasonal Naive", "seasonal_naive_prediction"),
        ("Global LightGBM", "lightgbm_prediction")
    ]:
        metrics = error_metrics(group, pred_col)

        segment_results.append({
            "segment": segment,
            "model": model_name,
            "n": len(group),
            **metrics
        })

segment_analysis = pd.DataFrame(segment_results)

# 6. Lưu kết quả
family_analysis.to_csv(
    TABLES_DIR / "lightgbm_error_by_family.csv",
    index=False
)

segment_analysis.to_csv(
    TABLES_DIR / "lightgbm_error_by_zero_sales.csv",
    index=False
)

# 7. Xuất bảng kết quả
print("BẢNG 1 — 10 NHÓM CÓ RMSLE CẢI THIỆN ÍT NHẤT")
display(
    family_analysis.sort_values(
        "RMSLE_improvement_pct"
    ).head(10).round(4)
)

print("\nBẢNG 2 — 10 NHÓM CÓ RMSLE CẢI THIỆN NHIỀU NHẤT")
display(
    family_analysis.sort_values(
        "RMSLE_improvement_pct",
        ascending=False
    ).head(10).round(4)
)

print("\nBẢNG 3 — SAI SỐ THEO DOANH SỐ BẰNG 0 / DƯƠNG")
display(segment_analysis.round(4))

print("\nSố nhóm sản phẩm LightGBM cải thiện MAE:",
      (family_analysis["MAE_improvement_pct"] > 0).sum())

print("Số nhóm sản phẩm LightGBM cải thiện RMSLE:",
      (family_analysis["RMSLE_improvement_pct"] > 0).sum())

print("\nPASS: Phân tích sai số hoàn tất.")
# CELL 21 — Phân rã sai số RMSLE

import numpy as np
import pandas as pd

df = analysis_df.copy()

# Bình phương sai số logarit của từng dự báo
df["baseline_sq_log_error"] = (
    np.log1p(df["seasonal_naive_prediction"])
    - np.log1p(df["sales"])
) ** 2

df["lightgbm_sq_log_error"] = (
    np.log1p(df["lightgbm_prediction"])
    - np.log1p(df["sales"])
) ** 2

# Chênh lệch dương nghĩa là LightGBM kém hơn
df["sq_log_error_difference"] = (
    df["lightgbm_sq_log_error"]
    - df["baseline_sq_log_error"]
)

df["sales_segment"] = np.where(
    df["sales"] == 0,
    "Zero sales",
    "Positive sales"
)

# 1. Phân rã theo trạng thái doanh số
segment_decomposition = (
    df.groupby("sales_segment")
    .agg(
        n=("id", "count"),
        baseline_sq_error_sum=("baseline_sq_log_error", "sum"),
        lightgbm_sq_error_sum=("lightgbm_sq_log_error", "sum"),
        difference_sum=("sq_log_error_difference", "sum")
    )
    .reset_index()
)

segment_decomposition["share_of_observations_pct"] = (
    segment_decomposition["n"] / len(df) * 100
)

# Đóng góp vào chênh lệch MSLE tổng thể
segment_decomposition["contribution_to_MSLE_difference"] = (
    segment_decomposition["difference_sum"] / len(df)
)

# 2. Phân rã theo nhóm sản phẩm
family_decomposition = (
    df.groupby("family")
    .agg(
        n=("id", "count"),
        zero_sales_pct=(
            "sales",
            lambda x: (x == 0).mean() * 100
        ),
        difference_sum=("sq_log_error_difference", "sum")
    )
    .reset_index()
)

family_decomposition["contribution_to_MSLE_difference"] = (
    family_decomposition["difference_sum"] / len(df)
)

family_decomposition = family_decomposition.sort_values(
    "contribution_to_MSLE_difference",
    ascending=False
)

# 3. Kiểm tra kết quả tổng thể
baseline_rmsle = np.sqrt(
    df["baseline_sq_log_error"].mean()
)

lightgbm_rmsle = np.sqrt(
    df["lightgbm_sq_log_error"].mean()
)

msle_difference = (
    df["sq_log_error_difference"].mean()
)

assert np.isclose(
    segment_decomposition[
        "contribution_to_MSLE_difference"
    ].sum(),
    msle_difference
)

# 4. Lưu bảng
segment_decomposition.to_csv(
    TABLES_DIR / "rmsle_decomposition_by_segment.csv",
    index=False
)

family_decomposition.to_csv(
    TABLES_DIR / "rmsle_decomposition_by_family.csv",
    index=False
)

# 5. Xuất kết quả
print("BẢNG 1 — ĐÓNG GÓP THEO TRẠNG THÁI DOANH SỐ")
display(segment_decomposition.round(6))

print("\nBẢNG 2 — 10 NHÓM LÀM MSLE TĂNG NHIỀU NHẤT")
display(family_decomposition.head(10).round(6))

print("\nKIỂM TRA TỔNG THỂ")
print("Baseline RMSLE:", round(baseline_rmsle, 6))
print("LightGBM RMSLE:", round(lightgbm_rmsle, 6))
print("Chênh lệch MSLE:", round(msle_difference, 6))

print("\nPASS: Phân rã sai số hoàn tất.")
# CELL 22 — Historical zero-sales profile

import pandas as pd
import numpy as np

VALIDATION_START = pd.Timestamp("2017-07-16")
LOOKBACK_DAYS = 90

history_start = VALIDATION_START - pd.Timedelta(
    days=LOOKBACK_DAYS
)

history_end = VALIDATION_START - pd.Timedelta(days=1)

history_90d = train_model_df.copy()
history_90d["date"] = pd.to_datetime(history_90d["date"])

history_90d = history_90d.loc[
    history_90d["date"].between(
        history_start, history_end
    )
]

# Tính tỷ lệ doanh số bằng 0 theo từng chuỗi
series_profile = (
    history_90d
    .groupby(["store_nbr", "family"])
    .agg(
        observed_days=("sales", "count"),
        zero_sales_days=("sales", lambda x: (x == 0).sum()),
        mean_sales=("sales", "mean")
    )
    .reset_index()
)

series_profile["zero_sales_ratio"] = (
    series_profile["zero_sales_days"]
    / series_profile["observed_days"]
)

# Kiểm tra dữ liệu
assert len(series_profile) == 54 * 33
assert series_profile["observed_days"].eq(90).all()
assert series_profile["zero_sales_ratio"].between(0, 1).all()

# Thống kê theo nhóm sản phẩm
family_profile = (
    series_profile.groupby("family")
    .agg(
        avg_zero_sales_ratio=("zero_sales_ratio", "mean"),
        median_zero_sales_ratio=("zero_sales_ratio", "median"),
        avg_sales=("mean_sales", "mean")
    )
    .reset_index()
    .sort_values(
        "avg_zero_sales_ratio",
        ascending=False
    )
)

# Phân bố chuỗi theo các ngưỡng thưa
threshold_summary = pd.DataFrame([
    {
        "threshold": threshold,
        "sparse_series": int(
            (series_profile["zero_sales_ratio"] >= threshold).sum()
        ),
        "total_series": len(series_profile),
        "sparse_pct": (
            (series_profile["zero_sales_ratio"] >= threshold).mean() * 100
        )
    }
    for threshold in [0.5, 0.7, 0.8, 0.9, 0.95]
])

# Lưu kết quả
series_profile.to_csv(
    TABLES_DIR / "historical_zero_sales_by_series.csv",
    index=False
)

family_profile.to_csv(
    TABLES_DIR / "historical_zero_sales_by_family.csv",
    index=False
)

threshold_summary.to_csv(
    TABLES_DIR / "historical_sparsity_thresholds.csv",
    index=False
)

print("BẢNG 1 — 10 NHÓM SẢN PHẨM CÓ DOANH SỐ THƯA NHẤT")
display(family_profile.head(10).round(4))

print("\nBẢNG 2 — PHÂN BỐ CHUỖI THEO NGƯỠNG THƯA")
display(threshold_summary.round(4))

print("\nSố chuỗi:", len(series_profile))
print("Khoảng lịch sử:", history_start.date(), "đến", history_end.date())
print("\nPASS: Historical zero-sales profile hoàn tất.")
# CELL 23 — Hybrid Forecasting với ngưỡng thưa 80%

import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error
)

SPARSITY_THRESHOLD = 0.80

# 1. Đọc kết quả dự báo và hồ sơ lịch sử
lgb_pred = pd.read_csv(
    PREDICTIONS_DIR / "validation_global_lightgbm.csv"
)

naive_pred = pd.read_csv(
    PREDICTIONS_DIR / "validation_seasonal_naive.csv"
)

series_profile = pd.read_csv(
    TABLES_DIR / "historical_zero_sales_by_series.csv"
)

# 2. Ghép hai mô hình theo ID
hybrid_df = lgb_pred.merge(
    naive_pred[["id", "seasonal_naive_prediction"]],
    on="id",
    how="inner",
    validate="one_to_one"
)

# 3. Ghép tỷ lệ doanh số bằng 0 trong lịch sử
hybrid_df = hybrid_df.merge(
    series_profile[
        ["store_nbr", "family", "zero_sales_ratio"]
    ],
    on=["store_nbr", "family"],
    how="left",
    validate="many_to_one"
)

# 4. Kiểm tra dữ liệu
assert len(hybrid_df) == 28512
assert hybrid_df["id"].is_unique
assert hybrid_df["zero_sales_ratio"].notna().all()

# 5. Xác định chuỗi thưa
hybrid_df["is_sparse"] = (
    hybrid_df["zero_sales_ratio"] >= SPARSITY_THRESHOLD
)

# 6. Tạo dự báo Hybrid
hybrid_df["hybrid_prediction"] = np.where(
    hybrid_df["is_sparse"],
    hybrid_df["seasonal_naive_prediction"],
    hybrid_df["lightgbm_prediction"]
)

# 7. Hàm tính chỉ số
def calculate_hybrid_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "RMSLE": np.sqrt(
            mean_squared_log_error(y_true, y_pred)
        )
    }

# 8. So sánh ba mô hình
results = []

for model_name, prediction_col in [
    ("Seasonal Naive", "seasonal_naive_prediction"),
    ("Global LightGBM", "lightgbm_prediction"),
    ("Hybrid 80%", "hybrid_prediction")
]:
    metrics = calculate_hybrid_metrics(
        hybrid_df["sales"],
        hybrid_df[prediction_col]
    )

    results.append({
        "model": model_name,
        **metrics
    })

hybrid_comparison = pd.DataFrame(results)

# 9. Tính mức cải thiện Hybrid so với LightGBM
lgb_metrics = hybrid_comparison.loc[
    hybrid_comparison["model"] == "Global LightGBM"
].iloc[0]

hybrid_metrics = hybrid_comparison.loc[
    hybrid_comparison["model"] == "Hybrid 80%"
].iloc[0]

hybrid_improvement = pd.DataFrame([
    {
        "metric": metric,
        "lightgbm": lgb_metrics[metric],
        "hybrid": hybrid_metrics[metric],
        "improvement_pct": (
            (lgb_metrics[metric] - hybrid_metrics[metric])
            / lgb_metrics[metric] * 100
        )
    }
    for metric in ["MAE", "RMSE", "RMSLE"]
])

# 10. Lưu kết quả
hybrid_df.to_csv(
    PREDICTIONS_DIR / "validation_hybrid_80.csv",
    index=False
)

hybrid_comparison.to_csv(
    TABLES_DIR / "hybrid_model_comparison.csv",
    index=False
)

hybrid_improvement.to_csv(
    TABLES_DIR / "hybrid_improvement_summary.csv",
    index=False
)

# 11. Hiển thị kết quả
print("BẢNG 1 — SO SÁNH BA MÔ HÌNH")
display(hybrid_comparison.round(4))

print("\nBẢNG 2 — HYBRID SO VỚI GLOBAL LIGHTGBM")
display(hybrid_improvement.round(4))

print("\nTHỐNG KÊ PHÂN LOẠI")
print(
    "Số chuỗi thưa:",
    hybrid_df.loc[
        hybrid_df["is_sparse"],
        ["store_nbr", "family"]
    ].drop_duplicates().shape[0]
)

print(
    "Số quan sát dùng Seasonal Naive:",
    int(hybrid_df["is_sparse"].sum())
)

print(
    "Số quan sát dùng LightGBM:",
    int((~hybrid_df["is_sparse"]).sum())
)

print("\nPASS: Hybrid Forecasting hoàn tất.")
# CELL 23 — Hybrid Forecasting với ngưỡng thưa 80%

import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error
)

SPARSITY_THRESHOLD = 0.80

# 1. Đọc kết quả dự báo và hồ sơ lịch sử
lgb_pred = pd.read_csv(
    PREDICTIONS_DIR / "validation_global_lightgbm.csv"
)

naive_pred = pd.read_csv(
    PREDICTIONS_DIR / "validation_seasonal_naive.csv"
)

series_profile = pd.read_csv(
    TABLES_DIR / "historical_zero_sales_by_series.csv"
)

# 2. Ghép hai mô hình theo ID
hybrid_df = lgb_pred.merge(
    naive_pred[["id", "seasonal_naive_prediction"]],
    on="id",
    how="inner",
    validate="one_to_one"
)

# 3. Ghép tỷ lệ doanh số bằng 0 trong lịch sử
hybrid_df = hybrid_df.merge(
    series_profile[
        ["store_nbr", "family", "zero_sales_ratio"]
    ],
    on=["store_nbr", "family"],
    how="left",
    validate="many_to_one"
)

# 4. Kiểm tra dữ liệu
assert len(hybrid_df) == 28512
assert hybrid_df["id"].is_unique
assert hybrid_df["zero_sales_ratio"].notna().all()

# 5. Xác định chuỗi thưa
hybrid_df["is_sparse"] = (
    hybrid_df["zero_sales_ratio"] >= SPARSITY_THRESHOLD
)

# 6. Tạo dự báo Hybrid
hybrid_df["hybrid_prediction"] = np.where(
    hybrid_df["is_sparse"],
    hybrid_df["seasonal_naive_prediction"],
    hybrid_df["lightgbm_prediction"]
)

# 7. Hàm tính chỉ số
def calculate_hybrid_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "RMSLE": np.sqrt(
            mean_squared_log_error(y_true, y_pred)
        )
    }

# 8. So sánh ba mô hình
results = []

for model_name, prediction_col in [
    ("Seasonal Naive", "seasonal_naive_prediction"),
    ("Global LightGBM", "lightgbm_prediction"),
    ("Hybrid 80%", "hybrid_prediction")
]:
    metrics = calculate_hybrid_metrics(
        hybrid_df["sales"],
        hybrid_df[prediction_col]
    )

    results.append({
        "model": model_name,
        **metrics
    })

hybrid_comparison = pd.DataFrame(results)

# 9. Tính mức cải thiện Hybrid so với LightGBM
lgb_metrics = hybrid_comparison.loc[
    hybrid_comparison["model"] == "Global LightGBM"
].iloc[0]

hybrid_metrics = hybrid_comparison.loc[
    hybrid_comparison["model"] == "Hybrid 80%"
].iloc[0]

hybrid_improvement = pd.DataFrame([
    {
        "metric": metric,
        "lightgbm": lgb_metrics[metric],
        "hybrid": hybrid_metrics[metric],
        "improvement_pct": (
            (lgb_metrics[metric] - hybrid_metrics[metric])
            / lgb_metrics[metric] * 100
        )
    }
    for metric in ["MAE", "RMSE", "RMSLE"]
])

# 10. Lưu kết quả
hybrid_df.to_csv(
    PREDICTIONS_DIR / "validation_hybrid_80.csv",
    index=False
)

hybrid_comparison.to_csv(
    TABLES_DIR / "hybrid_model_comparison.csv",
    index=False
)

hybrid_improvement.to_csv(
    TABLES_DIR / "hybrid_improvement_summary.csv",
    index=False
)

# 11. Hiển thị kết quả
print("BẢNG 1 — SO SÁNH BA MÔ HÌNH")
display(hybrid_comparison.round(4))

print("\nBẢNG 2 — HYBRID SO VỚI GLOBAL LIGHTGBM")
display(hybrid_improvement.round(4))

print("\nTHỐNG KÊ PHÂN LOẠI")
print(
    "Số chuỗi thưa:",
    hybrid_df.loc[
        hybrid_df["is_sparse"],
        ["store_nbr", "family"]
    ].drop_duplicates().shape[0]
)

print(
    "Số quan sát dùng Seasonal Naive:",
    int(hybrid_df["is_sparse"].sum())
)

print(
    "Số quan sát dùng LightGBM:",
    int((~hybrid_df["is_sparse"]).sum())
)

print("\nPASS: Hybrid Forecasting hoàn tất.")

# CELL 24 — Phân tích Hybrid theo nhóm sản phẩm

import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error
)

hybrid_df = pd.read_csv(
    PREDICTIONS_DIR / "validation_hybrid_80.csv"
)

def metrics(y, p):
    return {
        "MAE": mean_absolute_error(y, p),
        "RMSE": np.sqrt(mean_squared_error(y, p)),
        "RMSLE": np.sqrt(mean_squared_log_error(y, p))
    }

rows = []

for family, group in hybrid_df.groupby("family"):
    baseline = metrics(
        group["sales"],
        group["seasonal_naive_prediction"]
    )
    lightgbm = metrics(
        group["sales"],
        group["lightgbm_prediction"]
    )
    hybrid = metrics(
        group["sales"],
        group["hybrid_prediction"]
    )

    rows.append({
        "family": family,
        "n": len(group),
        "sparse_series": group.loc[
            group["is_sparse"], "store_nbr"
        ].nunique(),
        "baseline_RMSLE": baseline["RMSLE"],
        "lightgbm_RMSLE": lightgbm["RMSLE"],
        "hybrid_RMSLE": hybrid["RMSLE"],
        "hybrid_MAE": hybrid["MAE"],
        "hybrid_RMSLE_improvement_pct": (
            (lightgbm["RMSLE"] - hybrid["RMSLE"])
            / lightgbm["RMSLE"] * 100
            if lightgbm["RMSLE"] > 0 else np.nan
        )
    })

hybrid_family_analysis = pd.DataFrame(rows)

hybrid_family_analysis.to_csv(
    TABLES_DIR / "hybrid_error_by_family.csv",
    index=False
)

print("BẢNG 1 — NHÓM CẢI THIỆN RMSLE NHIỀU NHẤT")
display(
    hybrid_family_analysis.sort_values(
        "hybrid_RMSLE_improvement_pct",
        ascending=False
    ).head(10).round(4)
)

print("\nBẢNG 2 — NHÓM RMSLE CÒN CAO NHẤT")
display(
    hybrid_family_analysis.sort_values(
        "hybrid_RMSLE",
        ascending=False
    ).head(10).round(4)
)

print("\nPASS: Phân tích Hybrid theo family hoàn tất.")
# CELL 25 — Pre-check for recursive Hybrid forecasting

import pandas as pd
import numpy as np

required_objects = [
    "train_df",
    "model_validation_base",
    "lgb_model",
    "X_train",
    "FEATURES",
    "CAT_FEATURES",
    "PREDICTIONS_DIR",
    "TABLES_DIR"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Thiếu đối tượng: {missing_objects}"
)

# Đọc hồ sơ phân loại chuỗi từ lịch sử
series_profile = pd.read_csv(
    TABLES_DIR / "historical_zero_sales_by_series.csv"
)

series_profile["is_sparse"] = (
    series_profile["zero_sales_ratio"] >= 0.80
)

# Kiểm tra khóa định danh
assert not series_profile.duplicated(
    ["store_nbr", "family"]
).any()

assert len(series_profile) == 1782

# Kiểm tra số chuỗi thưa
n_sparse = int(series_profile["is_sparse"].sum())

assert n_sparse == 143

# Kiểm tra thời gian validation
validation_dates = pd.to_datetime(
    model_validation_base["date"]
)

assert validation_dates.min() == pd.Timestamp("2017-07-16")
assert validation_dates.max() == pd.Timestamp("2017-07-31")

# Kiểm tra mô hình
assert lgb_model.booster_.num_trees() == 1000

print("Số chuỗi:", len(series_profile))
print("Số chuỗi thưa:", n_sparse)
print("Số chuỗi dùng LightGBM:", len(series_profile) - n_sparse)
print("Số cây LightGBM:", lgb_model.booster_.num_trees())
print("Số đặc trưng:", len(FEATURES))
print("Số ngày validation:", validation_dates.nunique())

print("\nPASS: Sẵn sàng xây dựng Hybrid đệ quy.")
# CELL 26 — Recursive Hybrid Forecasting (16 ngày)

import pandas as pd
import numpy as np
import time

VALIDATION_START = pd.Timestamp("2017-07-16")
VALIDATION_END = pd.Timestamp("2017-07-31")
HISTORY_CUTOFF = VALIDATION_START - pd.Timedelta(days=1)
SPARSITY_THRESHOLD = 0.80

# 1. Lấy lịch sử thực tế trước validation
observed_history = train_df.loc[
    pd.to_datetime(train_df["date"]) <= HISTORY_CUTOFF,
    ["date", "store_nbr", "family", "sales"]
].copy()

observed_history["date"] = pd.to_datetime(
    observed_history["date"]
)

assert observed_history["date"].max() == HISTORY_CUTOFF

hybrid_history = observed_history.pivot(
    index="date",
    columns=["store_nbr", "family"],
    values="sales"
).asfreq("D")

# 2. Đọc danh sách chuỗi thưa
series_profile = pd.read_csv(
    TABLES_DIR / "historical_zero_sales_by_series.csv"
)

series_profile["is_sparse"] = (
    series_profile["zero_sales_ratio"] >= SPARSITY_THRESHOLD
)

assert len(series_profile) == 1782
assert series_profile["is_sparse"].sum() == 143

# 3. Chuẩn bị validation
validation_base = model_validation_base.copy()
validation_base["date"] = pd.to_datetime(
    validation_base["date"]
)

validation_base = validation_base.merge(
    series_profile[
        ["store_nbr", "family", "zero_sales_ratio", "is_sparse"]
    ],
    on=["store_nbr", "family"],
    how="left",
    validate="many_to_one"
)

assert validation_base["is_sparse"].notna().all()

# 4. Dự báo đệ quy
hybrid_batches = []
start_time = time.time()

for current_date in pd.date_range(
    VALIDATION_START,
    VALIDATION_END,
    freq="D"
):
    day_df = validation_base.loc[
        validation_base["date"] == current_date
    ].copy()

    past = hybrid_history.loc[
        :current_date - pd.Timedelta(days=1)
    ]

    # Tạo đặc trưng lag theo ngày lịch
    for lag in [7, 14, 28]:
        lag_date = current_date - pd.Timedelta(days=lag)

        lag_values = past.loc[lag_date]

        lag_lookup = lag_values.rename(
            f"lag_{lag}"
        ).reset_index()

        day_df = day_df.merge(
            lag_lookup,
            on=["store_nbr", "family"],
            how="left",
            validate="one_to_one"
        )

    # Tạo rolling mean
    for window in [7, 14, 28]:
        rolling_values = past.tail(window).mean(
            axis=0,
            skipna=True
        )

        rolling_lookup = rolling_values.rename(
            f"rolling_mean_{window}"
        ).reset_index()

        day_df = day_df.merge(
            rolling_lookup,
            on=["store_nbr", "family"],
            how="left",
            validate="one_to_one"
        )

    # Đồng bộ categorical features với tập train
    for col in CAT_FEATURES:
        day_df[col] = pd.Categorical(
            day_df[col],
            categories=X_train[col].cat.categories
        )

    # Dự báo bằng LightGBM
    lgb_predictions = np.maximum(
        lgb_model.predict(day_df[FEATURES]),
        0
    )

    day_df["lightgbm_prediction_recursive"] = (
        lgb_predictions
    )

    # Seasonal Naive: lấy doanh số đúng 7 ngày trước
    # Trong lịch sử chung, có thể là actual hoặc Hybrid prediction
    naive_date = current_date - pd.Timedelta(days=7)

    naive_values = past.loc[naive_date]

    naive_lookup = naive_values.rename(
        "seasonal_naive_recursive"
    ).reset_index()

    day_df = day_df.merge(
        naive_lookup,
        on=["store_nbr", "family"],
        how="left",
        validate="one_to_one"
    )

    assert day_df["seasonal_naive_recursive"].notna().all()

    # Chọn dự báo theo phân loại chuỗi
    day_df["hybrid_recursive_prediction"] = np.where(
        day_df["is_sparse"],
        day_df["seasonal_naive_recursive"],
        day_df["lightgbm_prediction_recursive"]
    )

    day_df["hybrid_recursive_prediction"] = (
        day_df["hybrid_recursive_prediction"].clip(lower=0)
    )

    # Lưu kết quả ngày
    hybrid_batches.append(
        day_df[
            [
                "id",
                "date",
                "store_nbr",
                "family",
                "sales",
                "is_sparse",
                "zero_sales_ratio",
                "lightgbm_prediction_recursive",
                "seasonal_naive_recursive",
                "hybrid_recursive_prediction"
            ]
        ].copy()
    )

    # Cập nhật lịch sử chung bằng dự báo HYBRID
    selected_predictions = day_df.set_index(
        ["store_nbr", "family"]
    )["hybrid_recursive_prediction"]

    hybrid_history.loc[current_date] = (
        selected_predictions.reindex(hybrid_history.columns)
    )

    print(
        f"Đã dự báo: {current_date.date()} "
        f"| {len(day_df)} chuỗi "
        f"| sparse: {day_df['is_sparse'].sum()}"
    )

# 5. Tổng hợp kết quả
hybrid_recursive_df = pd.concat(
    hybrid_batches,
    ignore_index=True
)

assert len(hybrid_recursive_df) == 28512
assert hybrid_recursive_df["id"].is_unique
assert hybrid_recursive_df["hybrid_recursive_prediction"].notna().all()
assert (hybrid_recursive_df["hybrid_recursive_prediction"] >= 0).all()

# 6. Lưu kết quả
output_path = (
    PREDICTIONS_DIR / "validation_hybrid_recursive_80.csv"
)

hybrid_recursive_df.to_csv(
    output_path,
    index=False
)

print("\nPASS: Recursive Hybrid Forecasting hoàn tất.")
print("Số dòng dự báo:", len(hybrid_recursive_df))
print(
    "Thời gian dự báo (giây):",
    round(time.time() - start_time, 2)
)
print("Đường dẫn lưu:", output_path)

display(hybrid_recursive_df.head())
# CELL 27 — Final comparison of forecasting approaches

import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error
)

# 1. Đọc kết quả dự báo
baseline = pd.read_csv(
    PREDICTIONS_DIR / "validation_seasonal_naive.csv"
)

lightgbm = pd.read_csv(
    PREDICTIONS_DIR / "validation_global_lightgbm.csv"
)

hybrid_static = pd.read_csv(
    PREDICTIONS_DIR / "validation_hybrid_80.csv"
)

hybrid_recursive = pd.read_csv(
    PREDICTIONS_DIR / "validation_hybrid_recursive_80.csv"
)

# 2. Ghép tất cả kết quả theo id
final_df = baseline[
    ["id", "sales", "seasonal_naive_prediction"]
].merge(
    lightgbm[["id", "lightgbm_prediction"]],
    on="id",
    how="inner",
    validate="one_to_one"
).merge(
    hybrid_static[["id", "hybrid_prediction"]],
    on="id",
    how="inner",
    validate="one_to_one"
).merge(
    hybrid_recursive[["id", "hybrid_recursive_prediction"]],
    on="id",
    how="inner",
    validate="one_to_one"
)

assert len(final_df) == 28512
assert final_df["id"].is_unique

# Kiểm tra giá trị thực tế giữa các file
for other_df in [lightgbm, hybrid_static, hybrid_recursive]:
    check = baseline[["id", "sales"]].merge(
        other_df[["id", "sales"]],
        on="id",
        suffixes=("_baseline", "_other"),
        validate="one_to_one"
    )
    assert len(check) == 28512
    assert np.allclose(
        check["sales_baseline"],
        check["sales_other"]
    )

# 3. Hàm đánh giá
def evaluate(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)

    assert np.isfinite(y).all()
    assert np.isfinite(p).all()
    assert (y >= 0).all()
    assert (p >= 0).all()

    return {
        "MAE": mean_absolute_error(y, p),
        "RMSE": np.sqrt(mean_squared_error(y, p)),
        "RMSLE": np.sqrt(mean_squared_log_error(y, p))
    }

# 4. So sánh bốn phương án
models = {
    "Seasonal Naive": "seasonal_naive_prediction",
    "Global LightGBM": "lightgbm_prediction",
    "Hybrid 80% (static)": "hybrid_prediction",
    "Hybrid 80% (recursive)": "hybrid_recursive_prediction"
}

results = []

for model_name, pred_col in models.items():
    metrics = evaluate(final_df["sales"], final_df[pred_col])
    results.append({
        "model": model_name,
        **metrics
    })

final_comparison = pd.DataFrame(results)

# 5. Mức cải thiện của Hybrid recursive so với baseline
baseline_metrics = final_comparison.iloc[0]
recursive_metrics = final_comparison.iloc[3]

improvement_vs_baseline = pd.DataFrame([
    {
        "metric": metric,
        "baseline": baseline_metrics[metric],
        "hybrid_recursive": recursive_metrics[metric],
        "improvement_pct": (
            (baseline_metrics[metric] - recursive_metrics[metric])
            / baseline_metrics[metric] * 100
        )
    }
    for metric in ["MAE", "RMSE", "RMSLE"]
])

# 6. Chênh lệch giữa Hybrid static và recursive
static_metrics = final_comparison.iloc[2]

static_vs_recursive = pd.DataFrame([
    {
        "metric": metric,
        "hybrid_static": static_metrics[metric],
        "hybrid_recursive": recursive_metrics[metric],
        "difference": (
            recursive_metrics[metric] - static_metrics[metric]
        )
    }
    for metric in ["MAE", "RMSE", "RMSLE"]
])

# 7. Lưu kết quả
final_comparison.to_csv(
    TABLES_DIR / "final_forecasting_model_comparison.csv",
    index=False
)

improvement_vs_baseline.to_csv(
    TABLES_DIR / "hybrid_recursive_vs_baseline.csv",
    index=False
)

static_vs_recursive.to_csv(
    TABLES_DIR / "hybrid_static_vs_recursive.csv",
    index=False
)

final_df.to_csv(
    PREDICTIONS_DIR / "validation_all_models.csv",
    index=False
)

# 8. Hiển thị
print("BẢNG 1 — SO SÁNH BỐN PHƯƠNG ÁN")
display(final_comparison.round(4))

print("\nBẢNG 2 — HYBRID RECURSIVE SO VỚI BASELINE")
display(improvement_vs_baseline.round(4))

print("\nBẢNG 3 — HYBRID STATIC SO VỚI RECURSIVE")
display(static_vs_recursive.round(6))

print("\nPASS: Final forecasting comparison hoàn tất.")
# CELL 28 — Recursive Seasonal Naive (leakage-free)

import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error
)

VALIDATION_START = pd.Timestamp("2017-07-16")
VALIDATION_END = pd.Timestamp("2017-07-31")

# 1. Chỉ lấy doanh số thực tế trước validation
history = train_df.loc[
    pd.to_datetime(train_df["date"]) < VALIDATION_START,
    ["date", "store_nbr", "family", "sales"]
].copy()

history["date"] = pd.to_datetime(history["date"])

history_wide = history.pivot(
    index="date",
    columns=["store_nbr", "family"],
    values="sales"
).asfreq("D")

assert history_wide.index.max() == pd.Timestamp("2017-07-15")
assert history_wide.shape[1] == 1782

# 2. Chuẩn bị validation
validation = model_validation_base[
    ["id", "date", "store_nbr", "family", "sales"]
].copy()

validation["date"] = pd.to_datetime(validation["date"])

# 3. Dự báo đệ quy
recursive_batches = []

for current_date in pd.date_range(
    VALIDATION_START, VALIDATION_END, freq="D"
):
    day_df = validation.loc[
        validation["date"] == current_date
    ].copy()

    reference_date = current_date - pd.Timedelta(days=7)

    # Dữ liệu thực tế nếu trước 16/07;
    # dự báo Seasonal Naive nếu nằm trong validation
    lag_values = history_wide.loc[reference_date]

    lag_lookup = lag_values.rename(
        "seasonal_naive_recursive_prediction"
    ).reset_index()

    day_df = day_df.merge(
        lag_lookup,
        on=["store_nbr", "family"],
        how="left",
        validate="one_to_one"
    )

    assert len(day_df) == 1782
    assert day_df[
        "seasonal_naive_recursive_prediction"
    ].notna().all()

    recursive_batches.append(day_df)

    # Cập nhật lịch sử bằng dự báo, KHÔNG dùng actual validation
    predicted_values = day_df.set_index(
        ["store_nbr", "family"]
    )["seasonal_naive_recursive_prediction"]

    history_wide.loc[current_date] = (
        predicted_values.reindex(history_wide.columns)
    )

recursive_naive_df = pd.concat(
    recursive_batches,
    ignore_index=True
)

assert len(recursive_naive_df) == 28512
assert recursive_naive_df["id"].is_unique

# 4. Đọc baseline cũ để kiểm tra sự khác biệt
old_naive = pd.read_csv(
    PREDICTIONS_DIR / "validation_seasonal_naive.csv"
)

comparison = recursive_naive_df.merge(
    old_naive[["id", "seasonal_naive_prediction"]],
    on="id",
    validate="one_to_one"
)

comparison["different"] = ~np.isclose(
    comparison["seasonal_naive_recursive_prediction"],
    comparison["seasonal_naive_prediction"]
)

comparison["period"] = np.where(
    comparison["date"] < pd.Timestamp("2017-07-23"),
    "Ngày 1–7",
    "Ngày 8–16"
)

leakage_check = comparison.groupby(
    "period"
).agg(
    observations=("id", "count"),
    different_predictions=("different", "sum")
).reset_index()

# 5. Tính chỉ số baseline mới
def calculate_metrics(y, p):
    return {
        "MAE": mean_absolute_error(y, p),
        "RMSE": np.sqrt(mean_squared_error(y, p)),
        "RMSLE": np.sqrt(mean_squared_log_error(y, p))
    }

metric_rows = []

for name, col in [
    ("Seasonal Naive cũ", "seasonal_naive_prediction"),
    ("Seasonal Naive đệ quy",
     "seasonal_naive_recursive_prediction")
]:
    metric_rows.append({
        "model": name,
        **calculate_metrics(comparison["sales"], comparison[col])
    })

naive_metrics = pd.DataFrame(metric_rows)

# 6. Lưu kết quả mới, KHÔNG ghi đè baseline cũ
recursive_naive_df.to_csv(
    PREDICTIONS_DIR / "validation_seasonal_naive_recursive.csv",
    index=False
)

leakage_check.to_csv(
    TABLES_DIR / "seasonal_naive_leakage_check.csv",
    index=False
)

naive_metrics.to_csv(
    TABLES_DIR / "seasonal_naive_recursive_metrics.csv",
    index=False
)

print("BẢNG 1 — KIỂM TRA SỰ KHÁC BIỆT BASELINE")
display(leakage_check)

print("\nBẢNG 2 — CHỈ SỐ BASELINE CŨ VÀ ĐỆ QUY")
display(naive_metrics.round(4))

print("\nPASS: Recursive Seasonal Naive hoàn tất.")
# CELL 29 — Official leakage-free model comparison

import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error
)

# 1. Đọc ba bộ dự báo đệ quy
naive = pd.read_csv(
    PREDICTIONS_DIR / "validation_seasonal_naive_recursive.csv"
)

lgb = pd.read_csv(
    PREDICTIONS_DIR / "validation_global_lightgbm.csv"
)

hybrid = pd.read_csv(
    PREDICTIONS_DIR / "validation_hybrid_recursive_80.csv"
)

# 2. Ghép theo ID
official_df = naive[
    ["id", "date", "store_nbr", "family", "sales",
     "seasonal_naive_recursive_prediction"]
].merge(
    lgb[["id", "sales", "lightgbm_prediction"]],
    on="id",
    how="inner",
    validate="one_to_one",
    suffixes=("", "_lgb")
).merge(
    hybrid[["id", "sales", "hybrid_recursive_prediction"]],
    on="id",
    how="inner",
    validate="one_to_one",
    suffixes=("", "_hybrid")
)

# 3. Kiểm tra dữ liệu
assert len(official_df) == 28512
assert official_df["id"].is_unique

assert np.allclose(
    official_df["sales"],
    official_df["sales_lgb"]
)

assert np.allclose(
    official_df["sales"],
    official_df["sales_hybrid"]
)

prediction_cols = [
    "seasonal_naive_recursive_prediction",
    "lightgbm_prediction",
    "hybrid_recursive_prediction"
]

assert official_df[prediction_cols].notna().all().all()
assert (official_df[prediction_cols] >= 0).all().all()

# 4. Tính metrics
def evaluate_model(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "RMSLE": np.sqrt(
            mean_squared_log_error(y_true, y_pred)
        )
    }

model_cols = {
    "Seasonal Naive (recursive)":
        "seasonal_naive_recursive_prediction",
    "Global LightGBM (recursive)":
        "lightgbm_prediction",
    "Hybrid 80% (recursive)":
        "hybrid_recursive_prediction"
}

results = []

for model_name, col in model_cols.items():
    results.append({
        "model": model_name,
        **evaluate_model(
            official_df["sales"],
            official_df[col]
        )
    })

official_comparison = pd.DataFrame(results)

# 5. Tính cải thiện so với baseline đệ quy
baseline_row = official_comparison.iloc[0]

improvement_rows = []

for _, row in official_comparison.iloc[1:].iterrows():
    for metric in ["MAE", "RMSE", "RMSLE"]:
        improvement_rows.append({
            "model": row["model"],
            "metric": metric,
            "baseline": baseline_row[metric],
            "model_value": row[metric],
            "improvement_pct": (
                (baseline_row[metric] - row[metric])
                / baseline_row[metric] * 100
            )
        })

official_improvement = pd.DataFrame(
    improvement_rows
)

# 6. Lưu kết quả chính thức
official_df.to_csv(
    PREDICTIONS_DIR / "validation_official_recursive_models.csv",
    index=False
)

official_comparison.to_csv(
    TABLES_DIR / "official_recursive_model_comparison.csv",
    index=False
)

official_improvement.to_csv(
    TABLES_DIR / "official_recursive_model_improvement.csv",
    index=False
)

# 7. Hiển thị
print("BẢNG 1 — SO SÁNH CHÍNH THỨC BA MÔ HÌNH")
display(official_comparison.round(4))

print("\nBẢNG 2 — CẢI THIỆN SO VỚI SEASONAL NAIVE ĐỆ QUY")
display(official_improvement.round(4))

print("\nPASS: Official leakage-free comparison hoàn tất.")
# CELL 30 — Simulation data readiness

import pandas as pd
import numpy as np

simulation_df = pd.read_csv(
    PREDICTIONS_DIR / "validation_official_recursive_models.csv"
)

simulation_df["date"] = pd.to_datetime(simulation_df["date"])

# 1. Kiểm tra cấu trúc
required_cols = [
    "id", "date", "store_nbr", "family", "sales",
    "seasonal_naive_recursive_prediction",
    "lightgbm_prediction",
    "hybrid_recursive_prediction"
]

assert set(required_cols).issubset(simulation_df.columns)
assert len(simulation_df) == 28512
assert simulation_df["id"].is_unique

# 2. Kiểm tra đầy đủ chuỗi theo ngày
daily_count = simulation_df.groupby("date").size()

assert daily_count.nunique() == 1
assert daily_count.iloc[0] == 1782

# 3. Kiểm tra giá trị
numeric_cols = [
    "sales",
    "seasonal_naive_recursive_prediction",
    "lightgbm_prediction",
    "hybrid_recursive_prediction"
]

assert simulation_df[numeric_cols].notna().all().all()
assert (simulation_df[numeric_cols] >= 0).all().all()

# 4. Thống kê nhu cầu thực tế
demand_summary = simulation_df["sales"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame("actual_sales")

# 5. Thống kê theo nhóm sản phẩm
family_demand = simulation_df.groupby("family").agg(
    total_sales=("sales", "sum"),
    mean_daily_sales=("sales", "mean"),
    median_daily_sales=("sales", "median"),
    zero_sales_ratio=("sales", lambda x: (x == 0).mean())
).reset_index()

family_demand = family_demand.sort_values(
    "total_sales", ascending=False
)

# 6. Lưu kết quả
demand_summary.to_csv(
    TABLES_DIR / "simulation_demand_summary.csv"
)

family_demand.to_csv(
    TABLES_DIR / "simulation_family_demand.csv",
    index=False
)

print("BẢNG 1 — THỐNG KÊ NHU CẦU THỰC TẾ")
display(demand_summary.round(3))

print("\nBẢNG 2 — 10 NHÓM CÓ TỔNG DOANH SỐ CAO NHẤT")
display(family_demand.head(10).round(3))

print("\nTHỐNG KÊ TỔNG QUÁT")
print("Số chuỗi:", simulation_df.groupby(
    ["store_nbr", "family"]
).ngroups)
print("Số ngày:", simulation_df["date"].nunique())
print("Tổng doanh số thực tế:", simulation_df["sales"].sum())

print("\nPASS: Simulation data readiness hoàn tất.")
# CELL 31 — Inventory simulation setup

import pandas as pd
import numpy as np

VALIDATION_START = pd.Timestamp("2017-07-16")
VALIDATION_END = pd.Timestamp("2017-07-31")

# 1. Đọc dữ liệu dùng cho mô phỏng
simulation_df = pd.read_csv(
    PREDICTIONS_DIR / "validation_official_recursive_models.csv",
    parse_dates=["date"]
)

# 2. Chỉ lấy lịch sử 28 ngày trước validation
train_dates = pd.to_datetime(train_df["date"])

history_28 = train_df.loc[
    (train_dates >= VALIDATION_START - pd.Timedelta(days=28))
    & (train_dates < VALIDATION_START),
    ["date", "store_nbr", "family", "sales"]
].copy()

history_28["date"] = pd.to_datetime(history_28["date"])

# Kiểm tra mỗi chuỗi đủ 28 ngày lịch sử
history_counts = history_28.groupby(
    ["store_nbr", "family"]
).size()

assert len(history_counts) == 1782
assert history_counts.eq(28).all()
assert history_28["date"].max() == pd.Timestamp("2017-07-15")

# 3. Tính độ lệch chuẩn từ 28 ngày lịch sử
historical_stats = history_28.groupby(
    ["store_nbr", "family"]
).agg(
    mean_sales_28d=("sales", "mean"),
    std_sales_28d=("sales", "std")
).reset_index()

# 4. Tính trung bình doanh số 7 ngày cuối
history_7 = history_28.loc[
    history_28["date"] >= (
        VALIDATION_START - pd.Timedelta(days=7)
    )
]

mean_7 = history_7.groupby(
    ["store_nbr", "family"]
).agg(
    mean_sales_7d=("sales", "mean")
).reset_index()

# 5. Lập bảng trạng thái ban đầu
initial_state = simulation_df[
    ["store_nbr", "family"]
].drop_duplicates().copy()

initial_state = initial_state.merge(
    historical_stats,
    on=["store_nbr", "family"],
    how="left",
    validate="one_to_one"
).merge(
    mean_7,
    on=["store_nbr", "family"],
    how="left",
    validate="one_to_one"
)

# Tồn kho ban đầu bằng 1.5 ngày doanh số trung bình
INITIAL_COVER_DAYS = 1.5

initial_state["initial_inventory"] = (
    INITIAL_COVER_DAYS * initial_state["mean_sales_7d"]
).clip(lower=0)

# Không giả định có hàng đang vận chuyển vào thời điểm đầu kỳ
initial_state["initial_pipeline"] = 0.0

# 6. Phân loại chuỗi thưa
series_profile = pd.read_csv(
    TABLES_DIR / "historical_zero_sales_by_series.csv"
)

initial_state = initial_state.merge(
    series_profile[
        ["store_nbr", "family", "zero_sales_ratio"]
    ],
    on=["store_nbr", "family"],
    how="left",
    validate="one_to_one"
)

initial_state["is_sparse"] = (
    initial_state["zero_sales_ratio"] >= 0.80
)

# 7. Thiết lập ba kịch bản
simulation_scenarios = pd.DataFrame([
    {
        "scenario": "S0_No_Safety",
        "safety_factor": 0.0
    },
    {
        "scenario": "S1_Base_Safety",
        "safety_factor": 1.0
    },
    {
        "scenario": "S2_High_Safety",
        "safety_factor": 2.0
    }
])

# 8. Tham số vận hành và chi phí giả định
simulation_parameters = pd.DataFrame([{
    "review_period_days": 1,
    "lead_time_days": 1,
    "initial_cover_days": INITIAL_COVER_DAYS,
    "holding_cost_per_unit_day": 1.0,
    "shortage_cost_per_unit": 5.0,
    "unmet_demand_policy": "lost_sales",
    "forecast_origin": "2017-07-15",
    "simulation_start": "2017-07-16",
    "simulation_end": "2017-07-31"
}])

# 9. Kiểm tra dữ liệu
assert len(initial_state) == 1782

check_cols = [
    "mean_sales_28d",
    "std_sales_28d",
    "mean_sales_7d",
    "initial_inventory",
    "zero_sales_ratio"
]

assert initial_state[check_cols].notna().all().all()
assert (initial_state["initial_inventory"] >= 0).all()
assert (initial_state["std_sales_28d"] >= 0).all()
assert initial_state["is_sparse"].sum() == 143

# 10. Lưu các tham số và trạng thái ban đầu
initial_state.to_csv(
    TABLES_DIR / "simulation_initial_inventory.csv",
    index=False
)

simulation_scenarios.to_csv(
    TABLES_DIR / "simulation_scenarios.csv",
    index=False
)

simulation_parameters.to_csv(
    TABLES_DIR / "simulation_parameters.csv",
    index=False
)

# 11. Hiển thị kết quả
print("BẢNG 1 — THỐNG KÊ TỒN KHO BAN ĐẦU")
display(
    initial_state[
        [
            "mean_sales_7d",
            "std_sales_28d",
            "initial_inventory"
        ]
    ].describe().round(3)
)

print("\nBẢNG 2 — CÁC KỊCH BẢN")
display(simulation_scenarios)

print("\nTHỐNG KÊ KIỂM TRA")
print("Số chuỗi:", len(initial_state))
print("Số chuỗi thưa:", int(initial_state["is_sparse"].sum()))
print("Số ngày lịch sử:", history_28["date"].nunique())
print(
    "Tổng tồn kho ban đầu giả định:",
    round(initial_state["initial_inventory"].sum(), 2)
)

print("\nPASS: Inventory simulation setup hoàn tất.")
# CELL 32 — Inventory Replenishment Simulation
# 3 forecasting models × 3 safety stock scenarios

import pandas as pd
import numpy as np
import time

start_time = time.time()

# =====================================================
# 1. ĐỌC DỮ LIỆU VÀ THAM SỐ
# =====================================================

simulation_df = pd.read_csv(
    PREDICTIONS_DIR / "validation_official_recursive_models.csv",
    parse_dates=["date"]
)

initial_state = pd.read_csv(
    TABLES_DIR / "simulation_initial_inventory.csv"
)

scenarios = pd.read_csv(
    TABLES_DIR / "simulation_scenarios.csv"
)

parameters = pd.read_csv(
    TABLES_DIR / "simulation_parameters.csv"
).iloc[0]

HOLDING_COST = float(
    parameters["holding_cost_per_unit_day"]
)

SHORTAGE_COST = float(
    parameters["shortage_cost_per_unit"]
)

assert int(parameters["lead_time_days"]) == 1
assert int(parameters["review_period_days"]) == 1
assert parameters["unmet_demand_policy"] == "lost_sales"

MODELS = {
    "Seasonal Naive":
        "seasonal_naive_recursive_prediction",

    "Global LightGBM":
        "lightgbm_prediction",

    "Hybrid 80%":
        "hybrid_recursive_prediction"
}

KEYS = ["store_nbr", "family"]

# =====================================================
# 2. CHUẨN BỊ DỮ LIỆU
# =====================================================

assert not initial_state.duplicated(KEYS).any()

simulation_df = simulation_df.merge(
    initial_state[
        KEYS + [
            "initial_inventory",
            "std_sales_28d",
            "is_sparse"
        ]
    ],
    on=KEYS,
    how="left",
    validate="many_to_one"
)

required_numeric = [
    "sales",
    "initial_inventory",
    "std_sales_28d"
] + list(MODELS.values())

assert simulation_df[required_numeric].notna().all().all()
assert np.isfinite(
    simulation_df[required_numeric].to_numpy(dtype=float)
).all()

assert (simulation_df[required_numeric] >= 0).all().all()

simulation_df = simulation_df.sort_values(
    ["date", "store_nbr", "family"]
).reset_index(drop=True)

dates = sorted(simulation_df["date"].unique())

assert len(dates) == 16
assert len(simulation_df) == 28512

# Tách dữ liệu theo ngày, bảo đảm cùng thứ tự chuỗi
day_frames = []

for current_date in dates:
    day = simulation_df.loc[
        simulation_df["date"] == current_date
    ].reset_index(drop=True)

    assert len(day) == 1782
    assert not day.duplicated(KEYS).any()

    day_frames.append(day)

template_keys = day_frames[0][KEYS]

for day in day_frames[1:]:
    assert day[KEYS].equals(template_keys)

N_SERIES = len(template_keys)
N_DAYS = len(day_frames)

initial_inventory = day_frames[0][
    "initial_inventory"
].to_numpy(dtype=float)

std_28d = day_frames[0][
    "std_sales_28d"
].to_numpy(dtype=float)

# =====================================================
# 3. CHẠY MÔ PHỎNG
# =====================================================

all_records = []

for model_name, forecast_col in MODELS.items():

    for _, scenario_row in scenarios.iterrows():

        scenario_name = scenario_row["scenario"]
        z = float(scenario_row["safety_factor"])

        # Mỗi phương án bắt đầu từ cùng tồn kho
        stock = initial_inventory.copy()

        # Không có đơn hàng đang vận chuyển đầu kỳ
        incoming_order = np.zeros(N_SERIES)

        safety_stock = z * std_28d

        for t, day in enumerate(day_frames):

            actual_demand = day["sales"].to_numpy(
                dtype=float
            )

            forecast_today = day[forecast_col].to_numpy(
                dtype=float
            )

            # A. Nhận hàng đầu ngày
            received = incoming_order.copy()

            available = stock + received

            # B. Đáp ứng nhu cầu
            fulfilled = np.minimum(
                available,
                actual_demand
            )

            unmet = np.maximum(
                actual_demand - fulfilled,
                0
            )

            # C. Tồn kho cuối ngày
            ending_inventory = np.maximum(
                available - fulfilled,
                0
            )

            # D. Đặt hàng để nhận đầu ngày tiếp theo
            if t < N_DAYS - 1:

                next_forecast = day_frames[t + 1][
                    forecast_col
                ].to_numpy(dtype=float)

                target_next_day = (
                    next_forecast + safety_stock
                )

                order_quantity = np.maximum(
                    target_next_day - ending_inventory,
                    0
                )

            else:
                # Không đặt hàng sau ngày cuối kỳ
                target_next_day = np.full(
                    N_SERIES, np.nan
                )

                order_quantity = np.zeros(N_SERIES)

            # E. Chi phí mô phỏng
            holding_cost = (
                ending_inventory * HOLDING_COST
            )

            shortage_cost = (
                unmet * SHORTAGE_COST
            )

            # F. Kiểm tra cân bằng dòng hàng
            assert np.allclose(
                fulfilled + unmet,
                actual_demand
            )

            assert np.allclose(
                available - fulfilled,
                ending_inventory
            )

            assert (ending_inventory >= 0).all()
            assert (order_quantity >= 0).all()

            # G. Ghi nhận từng ngày
            daily_result = pd.DataFrame({

                "model": model_name,
                "scenario": scenario_name,
                "safety_factor": z,

                "id": day["id"].to_numpy(),
                "date": day["date"].to_numpy(),
                "store_nbr": day["store_nbr"].to_numpy(),
                "family": day["family"].to_numpy(),
                "is_sparse": day["is_sparse"].to_numpy(),

                "forecast_today": forecast_today,
                "demand_proxy": actual_demand,

                "received_units": received,
                "available_before_demand": available,

                "fulfilled_units": fulfilled,
                "unmet_units": unmet,
                "ending_inventory": ending_inventory,

                "target_next_day": target_next_day,
                "order_units": order_quantity,

                "holding_cost_points": holding_cost,
                "shortage_cost_points": shortage_cost
            })

            all_records.append(daily_result)

            # H. Chuyển trạng thái sang ngày tiếp theo
            stock = ending_inventory.copy()
            incoming_order = order_quantity.copy()

        print(
            f"Hoàn thành: {model_name} | {scenario_name}"
        )

# =====================================================
# 4. TỔNG HỢP KẾT QUẢ
# =====================================================

inventory_records = pd.concat(
    all_records,
    ignore_index=True
)

assert len(inventory_records) == 28512 * 9

assert inventory_records.groupby(
    ["model", "scenario"]
).size().eq(28512).all()


def summarize_inventory(df, group_cols):

    summary = df.groupby(
        group_cols,
        as_index=False
    ).agg(
        total_demand=("demand_proxy", "sum"),
        fulfilled_units=("fulfilled_units", "sum"),
        unmet_units=("unmet_units", "sum"),

        total_ordered_units=("order_units", "sum"),

        avg_ending_inventory=(
            "ending_inventory", "mean"
        ),

        holding_cost_points=(
            "holding_cost_points", "sum"
        ),

        shortage_cost_points=(
            "shortage_cost_points", "sum"
        )
    )

    summary["fill_rate_pct"] = np.where(
        summary["total_demand"] > 0,
        summary["fulfilled_units"]
        / summary["total_demand"] * 100,
        np.nan
    )

    summary["total_cost_points"] = (
        summary["holding_cost_points"]
        + summary["shortage_cost_points"]
    )

    return summary


# 5. Tổng hợp toàn hệ thống
simulation_summary = summarize_inventory(
    inventory_records,
    ["scenario", "model"]
)

# 6. Phân tích riêng chuỗi thưa / không thưa
simulation_segment = summarize_inventory(
    inventory_records,
    ["scenario", "model", "is_sparse"]
)

# 7. Phân tích theo nhóm sản phẩm
simulation_family = summarize_inventory(
    inventory_records,
    ["scenario", "model", "family"]
)

# =====================================================
# 5. LƯU FILE
# =====================================================

inventory_records.to_csv(
    PREDICTIONS_DIR / "inventory_simulation_daily_records.csv",
    index=False
)

simulation_summary.to_csv(
    TABLES_DIR / "inventory_simulation_summary.csv",
    index=False
)

simulation_segment.to_csv(
    TABLES_DIR / "inventory_simulation_by_sparsity.csv",
    index=False
)

simulation_family.to_csv(
    TABLES_DIR / "inventory_simulation_by_family.csv",
    index=False
)

# =====================================================
# 6. HIỂN THỊ
# =====================================================

print("\nBẢNG 1 — KẾT QUẢ 9 PHƯƠNG ÁN MÔ PHỎNG")

display(
    simulation_summary[
        [
            "scenario",
            "model",
            "fill_rate_pct",
            "unmet_units",
            "avg_ending_inventory",
            "total_cost_points"
        ]
    ].round(2)
)

print("\nBẢNG 2 — KỊCH BẢN S1 THEO MỨC ĐỘ THƯA")

display(
    simulation_segment.loc[
        simulation_segment["scenario"] == "S1_Base_Safety",
        [
            "model",
            "is_sparse",
            "fill_rate_pct",
            "unmet_units",
            "avg_ending_inventory",
            "total_cost_points"
        ]
    ].round(2)
)

print("\nTHỐNG KÊ KIỂM TRA")
print("Số mô hình:", len(MODELS))
print("Số kịch bản:", len(scenarios))
print("Tổng phương án:", len(MODELS) * len(scenarios))
print("Tổng dòng mô phỏng:", len(inventory_records))
print("Số ngày:", N_DAYS)
print("Số chuỗi:", N_SERIES)
print(
    "Thời gian thực hiện (giây):",
    round(time.time() - start_time, 2)
)

print("\nPASS: Inventory simulation hoàn tất.")
# CELL 33 — Cost sensitivity & service-level trade-off

import pandas as pd
import numpy as np

# 1. Đọc kết quả Cell 32
summary = pd.read_csv(
    TABLES_DIR / "inventory_simulation_summary.csv"
)

segments = pd.read_csv(
    TABLES_DIR / "inventory_simulation_by_sparsity.csv"
)

segments["is_sparse"] = (
    segments["is_sparse"].astype(str).str.lower()
    .map({"true": True, "false": False})
)

assert len(summary) == 9
assert len(segments) == 18
assert segments["is_sparse"].notna().all()

SCENARIO = "S1_Base_Safety"

# 2. Phân rã chi phí kịch bản S1
s1 = summary.loc[
    summary["scenario"] == SCENARIO
].copy()

print("BẢNG 1 — PHÂN RÃ CHI PHÍ KỊCH BẢN S1")

display(
    s1[
        [
            "model",
            "holding_cost_points",
            "shortage_cost_points",
            "total_cost_points",
            "fill_rate_pct"
        ]
    ].round(2)
)

# 3. Phân tích riêng nhóm chuỗi thưa
sparse_s1 = segments.loc[
    (segments["scenario"] == SCENARIO)
    & (segments["is_sparse"] == True)
].copy()

sparse_idx = sparse_s1.set_index("model")

lgb = sparse_idx.loc["Global LightGBM"]
hybrid = sparse_idx.loc["Hybrid 80%"]

delta_unmet = (
    hybrid["unmet_units"] - lgb["unmet_units"]
)

delta_holding = (
    hybrid["holding_cost_points"]
    - lgb["holding_cost_points"]
)

delta_shortage_cost = (
    hybrid["shortage_cost_points"]
    - lgb["shortage_cost_points"]
)

delta_total_cost = (
    hybrid["total_cost_points"]
    - lgb["total_cost_points"]
)

tradeoff = pd.DataFrame([{
    "comparison": "Hybrid - LightGBM, sparse series",
    "additional_unmet_units": delta_unmet,
    "holding_cost_difference": delta_holding,
    "shortage_cost_difference": delta_shortage_cost,
    "total_cost_difference": delta_total_cost
}])

assert np.isclose(
    delta_holding + delta_shortage_cost,
    delta_total_cost,
    atol=0.01
)

print("\nBẢNG 2 — ĐÁNH ĐỔI Ở 143 CHUỖI THƯA")
display(tradeoff.round(2))

# 4. Phân tích độ nhạy chi phí thiếu hàng
# Giữ chi phí tồn kho = 1 điểm/đơn vị/ngày
# Thay đổi chi phí thiếu hàng từ 5 đến 100
penalties = [5, 20, 50, 100]

sensitivity_rows = []

for penalty in penalties:
    for _, row in s1.iterrows():
        adjusted_cost = (
            row["holding_cost_points"]
            + penalty * row["unmet_units"]
        )

        sensitivity_rows.append({
            "shortage_cost_per_unit": penalty,
            "model": row["model"],
            "adjusted_total_cost": adjusted_cost
        })

cost_sensitivity = pd.DataFrame(sensitivity_rows)

cost_matrix = cost_sensitivity.pivot(
    index="shortage_cost_per_unit",
    columns="model",
    values="adjusted_total_cost"
).reset_index()

print("\nBẢNG 3 — ĐỘ NHẠY VỚI CHI PHÍ THIẾU HÀNG")
display(cost_matrix.round(2))

# 5. Ngưỡng chi phí hòa vốn: Hybrid vs LightGBM
# Chỉ xét nhóm chuỗi thưa của S1
if delta_unmet > 0:
    break_even_penalty = (
        -delta_holding / delta_unmet
    )
else:
    break_even_penalty = np.nan

print(
    "\nChi phí thiếu hàng hòa vốn (S1, nhóm thưa):",
    round(break_even_penalty, 2)
    if np.isfinite(break_even_penalty)
    else "Không xác định"
)

# 6. Kiểm tra ràng buộc mức phục vụ minh họa
# Ngưỡng này là giả định phân tích, không phải quy định thực tế
MIN_OVERALL_FILL = 98.0
MIN_SPARSE_FILL = 90.0

sparse_rates = segments.loc[
    segments["is_sparse"] == True,
    ["scenario", "model", "fill_rate_pct"]
].rename(
    columns={"fill_rate_pct": "sparse_fill_rate_pct"}
)

service_check = summary.merge(
    sparse_rates,
    on=["scenario", "model"],
    how="left",
    validate="one_to_one"
)

service_check["meets_service_constraint"] = (
    (service_check["fill_rate_pct"] >= MIN_OVERALL_FILL)
    & (
        service_check["sparse_fill_rate_pct"]
        >= MIN_SPARSE_FILL
    )
)

print("\nBẢNG 4 — KIỂM TRA RÀNG BUỘC MỨC PHỤC VỤ")
display(
    service_check[
        [
            "scenario",
            "model",
            "fill_rate_pct",
            "sparse_fill_rate_pct",
            "total_cost_points",
            "meets_service_constraint"
        ]
    ].round(2)
)

# 7. Lưu kết quả
tradeoff.to_csv(
    TABLES_DIR / "inventory_sparse_tradeoff.csv",
    index=False
)

cost_sensitivity.to_csv(
    TABLES_DIR / "inventory_cost_sensitivity.csv",
    index=False
)

service_check.to_csv(
    TABLES_DIR / "inventory_service_constraints.csv",
    index=False
)

print("\nPASS: Cost sensitivity and service analysis hoàn tất.")
# CELL 34 — Sensitivity across all 9 policies

import pandas as pd
import numpy as np

summary = pd.read_csv(
    TABLES_DIR / "inventory_simulation_summary.csv"
)

segments = pd.read_csv(
    TABLES_DIR / "inventory_simulation_by_sparsity.csv"
)

# 1. Chuẩn hóa và ghép mức phục vụ nhóm thưa
segments["is_sparse"] = (
    segments["is_sparse"].astype(str).str.lower()
    .map({"true": True, "false": False})
)

sparse_fill = segments.loc[
    segments["is_sparse"] == True,
    ["scenario", "model", "fill_rate_pct"]
].rename(columns={
    "fill_rate_pct": "sparse_fill_rate_pct"
})

policies = summary.merge(
    sparse_fill,
    on=["scenario", "model"],
    how="left",
    validate="one_to_one"
)

assert len(policies) == 9
assert policies["sparse_fill_rate_pct"].notna().all()

# 2. Ràng buộc phục vụ giả định
MIN_OVERALL_FILL = 98.0
MIN_SPARSE_FILL = 90.0

policies["feasible"] = (
    (policies["fill_rate_pct"] >= MIN_OVERALL_FILL)
    & (policies["sparse_fill_rate_pct"] >= MIN_SPARSE_FILL)
)

# 3. Phân tích độ nhạy chi phí thiếu hàng
PENALTIES = [1, 5, 10, 20, 35, 50, 100]

sensitivity_list = []

for penalty in PENALTIES:
    temp = policies.copy()
    temp["shortage_penalty"] = penalty
    temp["adjusted_cost"] = (
        temp["holding_cost_points"]
        + penalty * temp["unmet_units"]
    )
    sensitivity_list.append(temp)

sensitivity_all = pd.concat(
    sensitivity_list,
    ignore_index=True
)

# 4. Kiểm tra kết quả tại mức chi phí ban đầu = 5
original = sensitivity_all.loc[
    sensitivity_all["shortage_penalty"] == 5
]

assert np.allclose(
    original["adjusted_cost"],
    original["total_cost_points"],
    atol=0.05
)

# 5. Phương án có chi phí thấp nhất, không ràng buộc
best_unconstrained = (
    sensitivity_all
    .sort_values(["shortage_penalty", "adjusted_cost"])
    .drop_duplicates("shortage_penalty")
    [["shortage_penalty", "scenario", "model",
      "adjusted_cost"]]
    .rename(columns={
        "scenario": "unconstrained_scenario",
        "model": "unconstrained_model",
        "adjusted_cost": "unconstrained_cost"
    })
)

# 6. Phương án có chi phí thấp nhất, có ràng buộc
best_feasible = (
    sensitivity_all.loc[sensitivity_all["feasible"]]
    .sort_values(["shortage_penalty", "adjusted_cost"])
    .drop_duplicates("shortage_penalty")
    [["shortage_penalty", "scenario", "model",
      "adjusted_cost"]]
    .rename(columns={
        "scenario": "feasible_scenario",
        "model": "feasible_model",
        "adjusted_cost": "feasible_cost"
    })
)

policy_selection = best_unconstrained.merge(
    best_feasible,
    on="shortage_penalty",
    how="left",
    validate="one_to_one"
)

# 7. Tính chi phí hòa vốn giữa S1 và S2
crossovers = []

for model, group in policies.groupby("model"):
    s1 = group.loc[
        group["scenario"] == "S1_Base_Safety"
    ].iloc[0]

    s2 = group.loc[
        group["scenario"] == "S2_High_Safety"
    ].iloc[0]

    demand_difference = (
        s1["unmet_units"] - s2["unmet_units"]
    )

    break_even = (
        (s2["holding_cost_points"]
         - s1["holding_cost_points"])
        / demand_difference
        if demand_difference > 0 else np.nan
    )

    crossovers.append({
        "model": model,
        "S1_S2_break_even_penalty": break_even
    })

crossover_df = pd.DataFrame(crossovers)

# 8. Lưu kết quả
sensitivity_all.to_csv(
    TABLES_DIR / "inventory_all_policy_sensitivity.csv",
    index=False
)

policy_selection.to_csv(
    TABLES_DIR / "inventory_policy_selection_by_penalty.csv",
    index=False
)

crossover_df.to_csv(
    TABLES_DIR / "inventory_S1_S2_break_even.csv",
    index=False
)

# 9. Hiển thị
print("BẢNG 1 — LỰA CHỌN CHÍNH SÁCH THEO CHI PHÍ THIẾU HÀNG")
display(policy_selection.round(2))

print("\nBẢNG 2 — NGƯỠNG HÒA VỐN S1 VÀ S2")
display(crossover_df.round(2))

print("\nBẢNG 3 — CÁC PHƯƠNG ÁN ĐẠT RÀNG BUỘC")
display(
    policies.loc[
        policies["feasible"],
        ["scenario", "model", "fill_rate_pct",
         "sparse_fill_rate_pct", "total_cost_points"]
    ].round(2)
)

print("\nPASS: Full policy sensitivity analysis hoàn tất.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
KIỂM TRA CẤU TRÚC DỰ ÁN
PROJECT_DIR: /content/drive/MyDrive/BusinessForecastingModels
PROJECT_DIR tồn tại: True
DATA_DIR: /content/drive/MyDrive/BusinessForecastingModels/Data
DATA_DIR tồn tại: True

DANH SÁCH TỆP TRONG DATA
[TỆP] holidays_events.csv (0.02 MB)
[TỆP] oil.csv (0.02 MB)
[TỆP] sample_submission.csv (0.33 MB)
[TỆP] stores.csv (0.00 MB)
[TỆP] test.csv (0.97 MB)
[TỆP] train.csv (116.16 MB)
[TỆP] transactions.csv (1.48 MB)

Hoàn tất kiểm tra đường dẫn.
Đang đọc: train.csv
  Kích thước: (3000888, 6)
Đang đọc: test.csv
  Kích thước: (28512, 5)
Đang đọc: stores.csv
  Kích thước: (54, 5)
Đang đọc: oil.csv
  Kích thước: (1218, 2)
Đang đọc: holidays_events.csv
  Kích thước: (350, 6)
Đang đọc: transactions.csv
  Kích thước: (83488, 3)
Đang đọc: sample_submission.csv
  Kích thước: (28512, 2)

Đã đọc thành công 7 tệp dữ liệu.
1. TỔNG QUAN DỮ LIỆU

TRAIN
Số dò

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0



TEST
Số dòng, số cột: (28512, 5)
Tên cột: ['id', 'date', 'store_nbr', 'family', 'onpromotion']


,id,date,store_nbr,family,onpromotion
0,3000888,2017-08-16,1,AUTOMOTIVE,0
1,3000889,2017-08-16,1,BABY CARE,0
2,3000890,2017-08-16,1,BEAUTY,2



STORES
Số dòng, số cột: (54, 5)
Tên cột: ['store_nbr', 'city', 'state', 'type', 'cluster']


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8


... (124 outputs left out)

,model,S1_S2_break_even_penalty
0,Global LightGBM,33.12
1,Hybrid 80%,33.11
2,Seasonal Naive,34.03



BẢNG 3 — CÁC PHƯƠNG ÁN ĐẠT RÀNG BUỘC


,scenario,model,fill_rate_pct,sparse_fill_rate_pct,total_cost_points
3,S1_Base_Safety,Global LightGBM,98.11,94.41,5056639.26
6,S2_High_Safety,Global LightGBM,98.91,95.43,8145177.67



PASS: Full policy sensitivity analysis hoàn tất.
